In [5]:
import socket
import subprocess
import time
import urllib.request
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False


streamlit_code = """
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

st.set_page_config(
    page_title="Streaming Marketing Analytics",
    page_icon="🎬",
    layout="wide"
)

# -----------------------------
# Load data
# -----------------------------
@st.cache_data
def load_data():
    df = pd.read_csv("final_with_predictions.csv")

    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    return df


df = load_data()

st.title("🎬 Streaming Marketing Analytics")
st.markdown("Dashboard para analizar visibilidad, engagement y valor comercial estimado de películas y series.")

# -----------------------------
# Sidebar filters
# -----------------------------
st.sidebar.header("Filtros")

content_options = sorted(df["content_type"].dropna().astype(str).unique().tolist()) if "content_type" in df.columns else []
selected_content = st.sidebar.multiselect(
    "Tipo de contenido",
    options=content_options,
    default=content_options
)

source_options = sorted(df["source"].dropna().astype(str).unique().tolist()) if "source" in df.columns else []
selected_source = st.sidebar.multiselect(
    "Fuente",
    options=source_options,
    default=source_options
)

language_options = sorted(df["original_language"].dropna().astype(str).unique().tolist()) if "original_language" in df.columns else []
selected_languages = st.sidebar.multiselect(
    "Idioma",
    options=language_options,
    default=language_options[:10] if len(language_options) > 10 else language_options
)

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider(
        "Rango de año",
        min_value=min_year,
        max_value=max_year,
        value=(min_year, max_year)
    )
else:
    year_range = None

# -----------------------------
# Apply filters
# -----------------------------
filtered_df = df.copy()

if selected_content and "content_type" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["content_type"].isin(selected_content)]

if selected_source and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"].isin(selected_source)]

if selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]

if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")
    ]

st.markdown(f"### Dataset filtrado: {filtered_df.shape[0]:,} títulos")

# -----------------------------
# KPIs
# -----------------------------
col1, col2, col3, col4 = st.columns(4)

total_titles = len(filtered_df)
avg_popularity = filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan
avg_vote = filtered_df["vote_average"].mean() if "vote_average" in filtered_df.columns else np.nan
avg_business = filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan

col1.metric("Total títulos", f"{total_titles:,}")
col2.metric("Promedio popularidad", f"{avg_popularity:.2f}" if pd.notna(avg_popularity) else "N/A")
col3.metric("Promedio rating", f"{avg_vote:.2f}" if pd.notna(avg_vote) else "N/A")
col4.metric("Promedio business value", f"{avg_business:.2f}" if pd.notna(avg_business) else "N/A")

# -----------------------------
# Tabs
# -----------------------------
tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
    "Overview",
    "Visibility",
    "Engagement",
    "Marketing Value",
    "Model / Clusters",
    "Dataset"
])

# -----------------------------
# TAB 1: OVERVIEW
# -----------------------------
with tab1:
    st.subheader("Resumen general")

    c1, c2 = st.columns(2)

    with c1:
        if "content_type" in filtered_df.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            filtered_df["content_type"].value_counts().plot(kind="bar", ax=ax)
            ax.set_title("Títulos por tipo de contenido")
            ax.set_xlabel("Content Type")
            ax.set_ylabel("Count")
            st.pyplot(fig)

    with c2:
        if "source" in filtered_df.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            filtered_df["source"].value_counts().plot(kind="bar", ax=ax)
            ax.set_title("Títulos por fuente")
            ax.set_xlabel("Source")
            ax.set_ylabel("Count")
            st.pyplot(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "popularity" in filtered_df.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(filtered_df["popularity"].dropna(), bins=30)
            ax.set_title("Distribución de popularidad")
            ax.set_xlabel("Popularity")
            ax.set_ylabel("Frequency")
            st.pyplot(fig)

    with c4:
        if "vote_average" in filtered_df.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(filtered_df["vote_average"].dropna(), bins=30)
            ax.set_title("Distribución de vote average")
            ax.set_xlabel("Vote Average")
            ax.set_ylabel("Frequency")
            st.pyplot(fig)

    if "release_year" in filtered_df.columns:
        yearly = filtered_df["release_year"].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(yearly.index, yearly.values)
        ax.set_title("Número de releases por año")
        ax.set_xlabel("Release Year")
        ax.set_ylabel("Titles")
        st.pyplot(fig)

# -----------------------------
# TAB 2: VISIBILITY
# -----------------------------
with tab2:
    st.subheader("Visibility Analysis")

    c1, c2 = st.columns(2)

    with c1:
        if "visibility_score" in filtered_df.columns:
            top_visibility = filtered_df[["title", "visibility_score"]].dropna().sort_values(
                by="visibility_score", ascending=False
            ).head(10)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(top_visibility["title"][::-1], top_visibility["visibility_score"][::-1])
            ax.set_title("Top 10 títulos por Visibility Score")
            ax.set_xlabel("Visibility Score")
            st.pyplot(fig)

    with c2:
        if "genre_names" in filtered_df.columns and "visibility_score" in filtered_df.columns:
            genre_df = filtered_df.copy()
            genre_df["genre_names"] = genre_df["genre_names"].fillna("Unknown").astype(str).str.split(", ")
            genre_df = genre_df.explode("genre_names")

            genre_visibility = genre_df.groupby("genre_names")["visibility_score"].mean().sort_values(ascending=False).head(10)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(genre_visibility.index[::-1], genre_visibility.values[::-1])
            ax.set_title("Top géneros por visibility")
            ax.set_xlabel("Average Visibility Score")
            st.pyplot(fig)

    if "visibility_score" in filtered_df.columns and "engagement_score" in filtered_df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(filtered_df["visibility_score"], filtered_df["engagement_score"], alpha=0.5)
        ax.set_title("Visibility vs Engagement")
        ax.set_xlabel("Visibility Score")
        ax.set_ylabel("Engagement Score")
        st.pyplot(fig)

# -----------------------------
# TAB 3: ENGAGEMENT
# -----------------------------
with tab3:
    st.subheader("Engagement Analysis")

    c1, c2 = st.columns(2)

    with c1:
        if "vote_count" in filtered_df.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(filtered_df["vote_count"].dropna(), bins=30)
            ax.set_title("Distribución de vote count")
            ax.set_xlabel("Vote Count")
            ax.set_ylabel("Frequency")
            st.pyplot(fig)

    with c2:
        if "engagement_score" in filtered_df.columns and "content_type" in filtered_df.columns:
            engagement_ct = filtered_df.groupby("content_type")["engagement_score"].mean()

            fig, ax = plt.subplots(figsize=(6, 4))
            engagement_ct.plot(kind="bar", ax=ax)
            ax.set_title("Engagement por tipo de contenido")
            ax.set_xlabel("Content Type")
            ax.set_ylabel("Average Engagement Score")
            st.pyplot(fig)

    if "genre_names" in filtered_df.columns and "engagement_score" in filtered_df.columns:
        genre_df = filtered_df.copy()
        genre_df["genre_names"] = genre_df["genre_names"].fillna("Unknown").astype(str).str.split(", ")
        genre_df = genre_df.explode("genre_names")

        genre_engagement = genre_df.groupby("genre_names")["engagement_score"].mean().sort_values(ascending=False).head(10)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(genre_engagement.index[::-1], genre_engagement.values[::-1])
        ax.set_title("Top géneros por engagement")
        ax.set_xlabel("Average Engagement Score")
        st.pyplot(fig)

# -----------------------------
# TAB 4: MARKETING VALUE
# -----------------------------
with tab4:
    st.subheader("Marketing Value")

    c1, c2 = st.columns(2)

    with c1:
        if "business_value_score" in filtered_df.columns:
            top_business = filtered_df[["title", "business_value_score"]].dropna().sort_values(
                by="business_value_score", ascending=False
            ).head(10)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(top_business["title"][::-1], top_business["business_value_score"][::-1])
            ax.set_title("Top 10 títulos por Business Value Score")
            ax.set_xlabel("Business Value Score")
            st.pyplot(fig)

    with c2:
        if "genre_names" in filtered_df.columns and "business_value_score" in filtered_df.columns:
            genre_df = filtered_df.copy()
            genre_df["genre_names"] = genre_df["genre_names"].fillna("Unknown").astype(str).str.split(", ")
            genre_df = genre_df.explode("genre_names")

            genre_business = genre_df.groupby("genre_names")["business_value_score"].mean().sort_values(ascending=False).head(10)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(genre_business.index[::-1], genre_business.values[::-1])
            ax.set_title("Top géneros por business value")
            ax.set_xlabel("Average Business Value Score")
            st.pyplot(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "marketing_segment" in filtered_df.columns:
            segment_counts = filtered_df["marketing_segment"].value_counts()

            fig, ax = plt.subplots(figsize=(7, 4))
            segment_counts.plot(kind="bar", ax=ax)
            ax.set_title("Distribución de marketing segments")
            ax.set_xlabel("Segment")
            ax.set_ylabel("Count")
            ax.tick_params(axis="x", rotation=30)
            st.pyplot(fig)

    with c4:
        if "predicted_business_value" in filtered_df.columns and "business_value_score" in filtered_df.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.scatter(filtered_df["business_value_score"], filtered_df["predicted_business_value"], alpha=0.5)
            ax.set_title("Actual vs Predicted Business Value")
            ax.set_xlabel("Actual")
            ax.set_ylabel("Predicted")
            st.pyplot(fig)

    if "release_year" in filtered_df.columns and "business_value_score" in filtered_df.columns:
        annual_business = filtered_df.groupby("release_year")["business_value_score"].mean().dropna()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(annual_business.index, annual_business.values)
        ax.set_title("Evolución anual del Business Value Score")
        ax.set_xlabel("Release Year")
        ax.set_ylabel("Average Business Value Score")
        st.pyplot(fig)

# -----------------------------
# TAB 5: MODEL / CLUSTERS
# -----------------------------
with tab5:
    st.subheader("Model and Clustering")

    c1, c2 = st.columns(2)

    with c1:
        if "cluster" in filtered_df.columns:
            cluster_counts = filtered_df["cluster"].value_counts().sort_index()

            fig, ax = plt.subplots(figsize=(6, 4))
            cluster_counts.plot(kind="bar", ax=ax)
            ax.set_title("Cantidad de títulos por cluster")
            ax.set_xlabel("Cluster")
            ax.set_ylabel("Count")
            st.pyplot(fig)

    with c2:
        if "cluster_label" in filtered_df.columns:
            cluster_label_counts = filtered_df["cluster_label"].value_counts()

            fig, ax = plt.subplots(figsize=(7, 4))
            cluster_label_counts.plot(kind="bar", ax=ax)
            ax.set_title("Distribución de cluster labels")
            ax.set_xlabel("Cluster Label")
            ax.set_ylabel("Count")
            ax.tick_params(axis="x", rotation=30)
            st.pyplot(fig)

    if {"visibility_score", "engagement_score", "cluster"}.issubset(filtered_df.columns):
        fig, ax = plt.subplots(figsize=(8, 5))
        scatter = ax.scatter(
            filtered_df["visibility_score"],
            filtered_df["engagement_score"],
            c=filtered_df["cluster"],
            alpha=0.6
        )
        ax.set_title("Clusters: Visibility vs Engagement")
        ax.set_xlabel("Visibility Score")
        ax.set_ylabel("Engagement Score")
        st.pyplot(fig)

    if "cluster_label" in filtered_df.columns and "business_value_score" in filtered_df.columns:
        cluster_summary = filtered_df.groupby("cluster_label")["business_value_score"].mean().sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(cluster_summary.index, cluster_summary.values)
        ax.set_title("Business Value promedio por cluster label")
        ax.set_xlabel("Cluster Label")
        ax.set_ylabel("Average Business Value Score")
        ax.tick_params(axis="x", rotation=30)
        st.pyplot(fig)

# -----------------------------
# TAB 6: DATASET
# -----------------------------
with tab6:
    st.subheader("Dataset Preview")

    st.dataframe(filtered_df.head(50), use_container_width=True)

    csv = filtered_df.to_csv(index=False).encode("utf-8-sig")
    st.download_button(
        label="Descargar dataset filtrado",
        data=csv,
        file_name="filtered_streaming_dataset.csv",
        mime="text/csv"
    )
"""

app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    subprocess.run(["open", "-a", "Safari", url])
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()

Starting Streamlit on http://127.0.0.1:55769 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:55769

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:55769


2026-03-23 16:32:57.406 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-23 16:33:16.720 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-23 16:33:49.231 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-23 16:34:25.897 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


In [6]:
import socket
import subprocess
import time
import urllib.request
import sys
import webbrowser
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False


streamlit_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Streaming Marketing Analytics",
    page_icon="🎬",
    layout="wide"
)

TOPIC_COLS = [
    "topic_lgbtq",
    "topic_politics",
    "topic_climate",
    "topic_war",
    "topic_family",
    "topic_crime",
    "topic_romance",
    "topic_technology",
]

TOPIC_LABELS = {
    "topic_lgbtq": "LGBTQ+",
    "topic_politics": "Politics",
    "topic_climate": "Climate",
    "topic_war": "War",
    "topic_family": "Family",
    "topic_crime": "Crime",
    "topic_romance": "Romance",
    "topic_technology": "Technology",
}

CARD_BG = "#1E293B"
FIG_BG = "#0F172A"
TEXT = "white"
MUTED = "#94A3B8"
GRID = "#334155"
ACCENT_1 = "#6366F1"
ACCENT_2 = "#10B981"
ACCENT_3 = "#F59E0B"
ACCENT_4 = "#EF4444"


# ──────────────────────────────────────────────────────────────────────────────
# STYLE
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
.block-container {
    padding-top: 1.5rem;
    padding-bottom: 2rem;
}
div[data-testid="stMetric"] {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    padding: 14px 16px;
    border-radius: 16px;
}
div[data-testid="stDataFrame"] {
    border-radius: 12px;
}
</style>
""", unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────────────────────────
# DATA
# ──────────────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("final_streaming_dataset.csv")

    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    if "release_year" in df.columns:
        df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

    numeric_cols = [
        "popularity", "vote_average", "vote_count",
        "visibility_score", "engagement_score",
        "business_value_score", "topic_diversity_score",
        "runtime_final", "cluster"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    text_cols = [
        "title", "overview", "overview_clean", "genre_names",
        "content_type", "source", "original_language",
        "marketing_segment", "cluster_label", "production_companies"
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    for col in TOPIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        else:
            df[col] = 0

    df["title_key"] = df["title"].fillna("").astype(str).str.lower().str.strip()

    return df


@st.cache_data
def prepare_similarity(df):
    work_df = df.copy()

    def topics_to_text(row):
        active = [TOPIC_LABELS[col] for col in TOPIC_COLS if row.get(col, 0) == 1]
        return " ".join(active)

    work_df["topics_text"] = work_df.apply(topics_to_text, axis=1)
    work_df["similarity_text"] = (
        work_df["title"].fillna("") + " "
        + work_df["genre_names"].fillna("").str.replace(",", " ", regex=False) + " "
        + work_df["overview_clean"].fillna("") + " "
        + work_df["overview"].fillna("") + " "
        + work_df["topics_text"].fillna("")
    ).str.lower()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )
    matrix = vectorizer.fit_transform(work_df["similarity_text"])

    return work_df, matrix


def get_active_topics(row):
    return [TOPIC_LABELS[col] for col in TOPIC_COLS if row.get(col, 0) == 1]


def get_similar_titles(df_similarity, matrix, selected_title, top_n=10):
    matches = df_similarity.index[df_similarity["title"] == selected_title].tolist()
    if not matches:
        return pd.DataFrame()

    idx = matches[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()

    sim_df = df_similarity.copy()
    sim_df["similarity_score"] = sims

    sim_df = sim_df[sim_df.index != idx].copy()

    rank_cols = [
        "title", "content_type", "genre_names", "release_year",
        "vote_average", "popularity", "visibility_score",
        "engagement_score", "business_value_score",
        "marketing_segment", "cluster_label", "similarity_score"
    ]
    existing_cols = [c for c in rank_cols if c in sim_df.columns]

    return sim_df.sort_values(
        by=["similarity_score", "business_value_score", "vote_average", "popularity"],
        ascending=[False, False, False, False]
    )[existing_cols].head(top_n)


def make_dark_fig(figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(FIG_BG)
    ax.set_facecolor(CARD_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT)
    return fig, ax


# ──────────────────────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────────────────────
df = load_data()
df_similarity, sim_matrix = prepare_similarity(df)

# ──────────────────────────────────────────────────────────────────────────────
# HEADER
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("### 🎬 Streaming Marketing Analytics")
st.markdown("**Dashboard de visibilidad, engagement, valor comercial y similitud entre títulos**")
st.divider()

# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────────────────────────────────────
st.sidebar.header("Filtros")

content_options = sorted([x for x in df["content_type"].dropna().unique().tolist() if x != ""])
selected_content = st.sidebar.multiselect(
    "Tipo de contenido",
    options=content_options,
    default=content_options
)

source_options = sorted([x for x in df["source"].dropna().unique().tolist() if x != ""])
selected_source = st.sidebar.multiselect(
    "Fuente",
    options=source_options,
    default=source_options
)

language_options = sorted([x for x in df["original_language"].dropna().unique().tolist() if x != ""])
selected_languages = st.sidebar.multiselect(
    "Idioma",
    options=language_options,
    default=language_options[:10] if len(language_options) > 10 else language_options
)

available_topics = [TOPIC_LABELS[c] for c in TOPIC_COLS if c in df.columns]
selected_topics = st.sidebar.multiselect(
    "Temas",
    options=available_topics,
    default=[]
)

title_query = st.sidebar.text_input("Buscar título", placeholder="Ej. The Rookie, War Machine...")

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider(
        "Rango de años",
        min_value=min_year,
        max_value=max_year,
        value=(min_year, max_year)
    )
else:
    year_range = None

# ──────────────────────────────────────────────────────────────────────────────
# FILTERS
# ──────────────────────────────────────────────────────────────────────────────
filtered_df = df.copy()

if selected_content and "content_type" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["content_type"].isin(selected_content)]

if selected_source and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"].isin(selected_source)]

if selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]

if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")
    ]

if title_query.strip():
    q = title_query.strip().lower()
    filtered_df = filtered_df[
        filtered_df["title"].str.lower().str.contains(q, na=False)
    ]

selected_topic_cols = [
    col for col, label in TOPIC_LABELS.items() if label in selected_topics
]
for col in selected_topic_cols:
    if col in filtered_df.columns:
        filtered_df = filtered_df[filtered_df[col] == 1]

st.markdown(f"### Dataset filtrado: {filtered_df.shape[0]:,} títulos")

# ──────────────────────────────────────────────────────────────────────────────
# KPIS
# ──────────────────────────────────────────────────────────────────────────────
total_titles = len(filtered_df)
avg_popularity = filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan
avg_vote = filtered_df["vote_average"].mean() if "vote_average" in filtered_df.columns else np.nan
avg_business = filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan
avg_visibility = filtered_df["visibility_score"].mean() if "visibility_score" in filtered_df.columns else np.nan

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total títulos", f"{total_titles:,}")
k2.metric("Popularidad promedio", f"{avg_popularity:.2f}" if pd.notna(avg_popularity) else "N/A")
k3.metric("Rating promedio", f"{avg_vote:.2f}" if pd.notna(avg_vote) else "N/A")
k4.metric("Business Value promedio", f"{avg_business:.2f}" if pd.notna(avg_business) else "N/A")

st.divider()

# ──────────────────────────────────────────────────────────────────────────────
# TABS
# ──────────────────────────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4 = st.tabs([
    "📊 Overview",
    "🎯 Marketing Performance",
    "🔎 Similar Titles Finder",
    "🗂 Dataset Explorer"
])

# ══════════════════════════════════════════════════════════════════════════════
# TAB 1 – OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════
with tab1:
    st.subheader("Vista general del catálogo")

    c1, c2 = st.columns(2)

    with c1:
        if "content_type" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["content_type"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_1)
            ax.set_title("Títulos por tipo de contenido", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Cantidad", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "source" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["source"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_2)
            ax.set_title("Títulos por fuente", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Cantidad", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "popularity" in filtered_df.columns and filtered_df["popularity"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["popularity"].dropna(), bins=30, color=ACCENT_3)
            ax.set_title("Distribución de popularidad", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Popularity", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c4:
        if "vote_average" in filtered_df.columns and filtered_df["vote_average"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["vote_average"].dropna(), bins=30, color=ACCENT_1)
            ax.set_title("Distribución de rating", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Vote Average", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    if "release_year" in filtered_df.columns and filtered_df["release_year"].notna().any():
        yearly = filtered_df["release_year"].dropna().astype(int).value_counts().sort_index()
        fig, ax = make_dark_fig((12, 4))
        ax.plot(yearly.index, yearly.values, color=ACCENT_2, linewidth=2.5)
        ax.set_title("Evolución de títulos por año", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        ax.set_xlabel("Año de estreno", color=TEXT)
        ax.set_ylabel("Cantidad", color=TEXT)
        st.pyplot(fig)
        plt.close(fig)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 2 – MARKETING PERFORMANCE
# ══════════════════════════════════════════════════════════════════════════════
with tab2:
    st.subheader("Análisis de marketing y performance")

    highlight_metric = st.radio(
        "Métrica destacada:",
        options=["visibility_score", "engagement_score", "business_value_score"],
        horizontal=True,
        index=2,
        format_func=lambda x: {
            "visibility_score": "Visibility",
            "engagement_score": "Engagement",
            "business_value_score": "Business Value"
        }[x]
    )

    c1, c2 = st.columns(2)

    with c1:
        if highlight_metric in filtered_df.columns and "title" in filtered_df.columns and not filtered_df.empty:
            top_df = filtered_df[["title", highlight_metric]].dropna().sort_values(
                by=highlight_metric, ascending=False
            ).head(10)

            fig, ax = make_dark_fig((8, 5))
            ax.barh(top_df["title"][::-1], top_df[highlight_metric][::-1], color=ACCENT_1)
            ax.set_title(f"Top 10 por {highlight_metric}", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "marketing_segment" in filtered_df.columns and not filtered_df.empty:
            seg = filtered_df["marketing_segment"].value_counts()
            fig, ax = make_dark_fig((8, 5))
            ax.bar(seg.index, seg.values, color=ACCENT_2)
            ax.set_title("Distribución por marketing segment", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)

    with c3:
        if {"visibility_score", "engagement_score"}.issubset(filtered_df.columns) and not filtered_df.empty:
            fig, ax = make_dark_fig((8, 5))
            ax.scatter(
                filtered_df["visibility_score"],
                filtered_df["engagement_score"],
                alpha=0.5,
                color=ACCENT_3
            )
            ax.set_title("Visibility vs Engagement", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Visibility Score", color=TEXT)
            ax.set_ylabel("Engagement Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c4:
        if "cluster_label" in filtered_df.columns and "business_value_score" in filtered_df.columns and not filtered_df.empty:
            summary = filtered_df.groupby("cluster_label")["business_value_score"].mean().sort_values(ascending=False)
            fig, ax = make_dark_fig((8, 5))
            ax.bar(summary.index, summary.values, color=ACCENT_4)
            ax.set_title("Business Value por cluster label", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    with st.expander("Ver tabla resumen de métricas"):
        cols = [c for c in [
            "title", "content_type", "genre_names", "release_year",
            "popularity", "vote_average", "visibility_score",
            "engagement_score", "business_value_score",
            "marketing_segment", "cluster_label"
        ] if c in filtered_df.columns]

        table_df = filtered_df[cols].sort_values(
            by=[c for c in ["business_value_score", "visibility_score", "vote_average"] if c in filtered_df.columns],
            ascending=False
        ).head(25)

        st.dataframe(table_df, use_container_width=True)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 3 – SIMILAR TITLES FINDER
# ══════════════════════════════════════════════════════════════════════════════
with tab3:
    st.subheader("Buscador de títulos similares")

    available_titles = sorted(filtered_df["title"].dropna().astype(str).unique().tolist())

    if not available_titles:
        st.warning("No hay títulos disponibles con los filtros actuales.")
    else:
        selected_title = st.selectbox(
            "Selecciona una película o serie",
            options=available_titles
        )

        selected_row = filtered_df[filtered_df["title"] == selected_title].head(1)

        if not selected_row.empty:
            row = selected_row.iloc[0]

            # Rankings globales
            global_df = df.copy()

            def get_rank(col, title):
                if col not in global_df.columns:
                    return None, None
                rank_df = global_df[["title", col]].dropna().sort_values(by=col, ascending=False).reset_index(drop=True)
                rank_df.index = rank_df.index + 1
                matches = rank_df[rank_df["title"] == title]
                if matches.empty:
                    return None, len(rank_df)
                return int(matches.index[0]), len(rank_df)

            rank_pop, total_pop = get_rank("popularity", selected_title)
            rank_vote, total_vote = get_rank("vote_average", selected_title)
            rank_business, total_business = get_rank("business_value_score", selected_title)

            st.markdown(f"#### {selected_title}")

            info1, info2, info3, info4 = st.columns(4)
            info1.metric("Tipo", row.get("content_type", "N/A"))
            info2.metric("Año", int(row["release_year"]) if pd.notna(row.get("release_year")) else "N/A")
            info3.metric("Rating", f"{row['vote_average']:.2f}" if pd.notna(row.get("vote_average")) else "N/A")
            info4.metric("Business Value", f"{row['business_value_score']:.2f}" if pd.notna(row.get("business_value_score")) else "N/A")

            st.markdown(
                f"**Géneros:** {row.get('genre_names', 'N/A')}  \n"
                f"**Marketing Segment:** {row.get('marketing_segment', 'N/A')}  \n"
                f"**Cluster:** {row.get('cluster_label', 'N/A')}"
            )

            active_topics = get_active_topics(row)
            st.markdown(f"**Temas detectados:** {', '.join(active_topics) if active_topics else 'No detectados'}")

            if row.get("overview"):
                st.markdown("**Overview**")
                st.write(row["overview"])

            r1, r2, r3 = st.columns(3)
            r1.metric("Ranking popularidad", f"#{rank_pop}" if rank_pop else "N/A", delta=f"de {total_pop}" if total_pop else None)
            r2.metric("Ranking rating", f"#{rank_vote}" if rank_vote else "N/A", delta=f"de {total_vote}" if total_vote else None)
            r3.metric("Ranking business value", f"#{rank_business}" if rank_business else "N/A", delta=f"de {total_business}" if total_business else None)

            st.divider()

            similar_df = get_similar_titles(df_similarity, sim_matrix, selected_title, top_n=10)

            st.markdown("#### Títulos similares")

            if similar_df.empty:
                st.info("No se encontraron títulos similares.")
            else:
                fig, ax = make_dark_fig((9, 5))
                plot_df = similar_df.head(8).copy()
                ax.barh(
                    plot_df["title"][::-1],
                    plot_df["similarity_score"][::-1],
                    color=ACCENT_1
                )
                ax.set_title("Top similitud", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                ax.set_xlabel("Similarity Score", color=TEXT)
                st.pyplot(fig)
                plt.close(fig)

                show_cols = [c for c in [
                    "title", "content_type", "genre_names", "release_year",
                    "vote_average", "popularity", "visibility_score",
                    "engagement_score", "business_value_score",
                    "marketing_segment", "cluster_label", "similarity_score"
                ] if c in similar_df.columns]

                display_df = similar_df[show_cols].copy()
                if "similarity_score" in display_df.columns:
                    display_df["similarity_score"] = display_df["similarity_score"].round(3)

                st.dataframe(display_df, use_container_width=True)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 4 – DATASET EXPLORER
# ══════════════════════════════════════════════════════════════════════════════
with tab4:
    st.subheader("Explorador del dataset")

    explorer_cols = [c for c in [
        "title", "content_type", "genre_names", "release_year",
        "original_language", "source", "popularity", "vote_average",
        "vote_count", "visibility_score", "engagement_score",
        "business_value_score", "marketing_segment", "cluster_label",
        "overview"
    ] if c in filtered_df.columns]

    st.dataframe(filtered_df[explorer_cols].head(100), use_container_width=True)

    csv = filtered_df.to_csv(index=False).encode("utf-8-sig")
    st.download_button(
        label="Descargar dataset filtrado",
        data=csv,
        file_name="filtered_streaming_dataset.csv",
        mime="text/csv"
    )
'''

app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    webbrowser.open(url)
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()

Starting Streamlit on http://127.0.0.1:54843 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:54843

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:54843


2026-03-24 01:05:04.126 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 01:05:05.335 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 01:05:05.339 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 01:05:40.561 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 01:05:41.823 

In [7]:
import socket
import subprocess
import time
import urllib.request
import sys
import webbrowser
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False


streamlit_code = r'''
import re
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Streaming Marketing Analytics",
    page_icon="🎬",
    layout="wide"
)

CARD_BG = "#1E293B"
FIG_BG = "#0F172A"
TEXT = "white"
MUTED = "#94A3B8"
ACCENT_1 = "#6366F1"
ACCENT_2 = "#10B981"
ACCENT_3 = "#F59E0B"
ACCENT_4 = "#EF4444"

TOPIC_COLS = [
    "topic_lgbtq",
    "topic_politics",
    "topic_climate",
    "topic_war",
    "topic_family",
    "topic_crime",
    "topic_romance",
    "topic_technology",
]

TOPIC_LABELS = {
    "topic_lgbtq": "LGBTQ+",
    "topic_politics": "Politics",
    "topic_climate": "Climate",
    "topic_war": "War",
    "topic_family": "Family",
    "topic_crime": "Crime",
    "topic_romance": "Romance",
    "topic_technology": "Technology",
}

TOPIC_KEYWORDS = {
    "LGBTQ+": ["gay", "lesbian", "lgbt", "trans", "queer", "bisexual", "drag"],
    "Politics": ["politics", "president", "government", "election", "policy", "minister", "senate", "congress"],
    "Climate": ["climate", "environment", "global warming", "pollution", "sustainability", "ecology"],
    "War": ["war", "battle", "army", "soldier", "military", "conflict", "invasion"],
    "Family": ["family", "mother", "father", "children", "home", "parent", "siblings"],
    "Crime": ["crime", "murder", "police", "detective", "investigation", "gang", "mafia", "robbery"],
    "Romance": ["love", "romance", "relationship", "couple", "marriage", "affair"],
    "Technology": ["technology", "ai", "robot", "future", "cyber", "machine", "algorithm"],
    "Mental Health": ["depression", "anxiety", "trauma", "therapy", "mental", "psychological"],
    "Coming of Age": ["teen", "adolescent", "growing up", "school", "youth", "friendship"],
    "Social Issues": ["racism", "inequality", "poverty", "discrimination", "justice", "migration"],
    "Fantasy / Supernatural": ["magic", "witch", "dragon", "supernatural", "monster", "curse", "fantasy"],
}


# ──────────────────────────────────────────────────────────────────────────────
# STYLE
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
.block-container {
    padding-top: 1.4rem;
    padding-bottom: 2rem;
}
div[data-testid="stMetric"] {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    padding: 14px 16px;
    border-radius: 16px;
}
div[data-testid="stDataFrame"] {
    border-radius: 12px;
}
</style>
""", unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9áéíóúñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def detect_topics_from_synopsis(text):
    text_clean = clean_text(text)
    found_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword.lower() in text_clean for keyword in keywords):
            found_topics.append(topic)

    if not found_topics:
        found_topics.append("General / No clear topic")

    return found_topics


def get_active_topics_from_row(row):
    return [TOPIC_LABELS[col] for col in TOPIC_COLS if row.get(col, 0) == 1]


def make_dark_fig(figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(FIG_BG)
    ax.set_facecolor(CARD_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT)
    return fig, ax


# ──────────────────────────────────────────────────────────────────────────────
# DATA
# ──────────────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("final_streaming_dataset.csv")

    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    if "release_year" in df.columns:
        df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

    numeric_cols = [
        "popularity", "vote_average", "vote_count",
        "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value",
        "topic_diversity_score", "runtime_final", "cluster"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    text_cols = [
        "title", "overview", "overview_clean", "genre_names",
        "content_type", "source", "original_language",
        "marketing_segment", "cluster_label", "production_companies"
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    for col in TOPIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        else:
            df[col] = 0

    df["title_key"] = df["title"].fillna("").astype(str).str.lower().str.strip()

    if "overview_clean" not in df.columns:
        df["overview_clean"] = df["overview"].fillna("").apply(clean_text)
    else:
        df["overview_clean"] = df["overview_clean"].fillna("").apply(clean_text)

    return df


@st.cache_data
def prepare_similarity(df):
    work_df = df.copy()

    work_df["topics_text"] = work_df.apply(
        lambda row: " ".join(get_active_topics_from_row(row)),
        axis=1
    )

    work_df["similarity_text"] = (
        work_df["title"].fillna("") + " "
        + work_df["genre_names"].fillna("").str.replace(",", " ", regex=False) + " "
        + work_df["overview_clean"].fillna("") + " "
        + work_df["overview"].fillna("") + " "
        + work_df["topics_text"].fillna("")
    ).str.lower()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )
    matrix = vectorizer.fit_transform(work_df["similarity_text"])

    return work_df, matrix


@st.cache_resource
def build_similarity_model(df):
    model_df = df.copy()

    if "overview" not in model_df.columns:
        model_df["overview"] = ""

    model_df["overview_clean"] = model_df["overview"].fillna("").apply(clean_text)

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform(model_df["overview_clean"])

    return vectorizer, tfidf_matrix, model_df


def get_similar_titles(df_similarity, matrix, selected_title, top_n=10):
    matches = df_similarity.index[df_similarity["title"] == selected_title].tolist()
    if not matches:
        return pd.DataFrame()

    idx = matches[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()

    sim_df = df_similarity.copy()
    sim_df["similarity_score"] = sims
    sim_df = sim_df[sim_df.index != idx].copy()

    rank_cols = [
        "title", "content_type", "genre_names", "release_year",
        "vote_average", "popularity", "visibility_score",
        "engagement_score", "business_value_score",
        "predicted_business_value", "marketing_segment",
        "cluster_label", "similarity_score"
    ]
    existing_cols = [c for c in rank_cols if c in sim_df.columns]

    return sim_df.sort_values(
        by=["similarity_score", "business_value_score", "vote_average", "popularity"],
        ascending=[False, False, False, False]
    )[existing_cols].head(top_n)


def get_top_similar_titles(user_synopsis, content_type, df, top_n=3):
    vectorizer, tfidf_matrix, model_df = build_similarity_model(df)

    filtered = model_df.copy()

    if "content_type" in filtered.columns and content_type and content_type.lower() != "all":
        filtered = filtered[
            filtered["content_type"].astype(str).str.lower() == content_type.lower()
        ].copy()

    if filtered.empty:
        return pd.DataFrame()

    filtered_indices = filtered.index.tolist()

    user_text = clean_text(user_synopsis)
    user_vec = vectorizer.transform([user_text])

    similarities = cosine_similarity(user_vec, tfidf_matrix[filtered_indices]).flatten()
    filtered["text_similarity"] = similarities

    user_topics = set(detect_topics_from_synopsis(user_synopsis))

    if "overview" in filtered.columns:
        filtered["detected_topics_temp"] = filtered["overview"].fillna("").apply(detect_topics_from_synopsis)

        def topic_overlap_score(topics_list):
            item_topics = set(topics_list)
            if not user_topics or not item_topics:
                return 0
            return len(user_topics.intersection(item_topics)) / max(len(user_topics), 1)

        filtered["topic_similarity"] = filtered["detected_topics_temp"].apply(topic_overlap_score)
    else:
        filtered["topic_similarity"] = 0

    filtered["similarity_score"] = (
        0.8 * filtered["text_similarity"] +
        0.2 * filtered["topic_similarity"]
    )

    cols_to_show = [
        col for col in [
            "title",
            "content_type",
            "genre_names",
            "original_language",
            "release_year",
            "overview",
            "business_value_score",
            "predicted_business_value",
            "text_similarity",
            "topic_similarity",
            "similarity_score"
        ] if col in filtered.columns
    ]

    result = filtered.sort_values("similarity_score", ascending=False).head(top_n)
    return result[cols_to_show]


def get_rank(df_all, title, col):
    if col not in df_all.columns:
        return None, None

    rank_df = df_all[["title", col]].dropna().sort_values(by=col, ascending=False).reset_index(drop=True)
    rank_df.index = rank_df.index + 1
    matches = rank_df[rank_df["title"] == title]

    if matches.empty:
        return None, len(rank_df)

    return int(matches.index[0]), len(rank_df)


# ──────────────────────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────────────────────
df = load_data()
df_similarity, sim_matrix = prepare_similarity(df)

# ──────────────────────────────────────────────────────────────────────────────
# HEADER
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("### 🎬 Streaming Marketing Analytics")
st.markdown("**Dashboard de visibilidad, engagement, valor comercial y similitud entre títulos**")
st.divider()

# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────────────────────────────────────
st.sidebar.header("Filtros")

content_options_real = sorted([x for x in df["content_type"].dropna().unique().tolist() if x != ""])
content_options = ["All"] + content_options_real
selected_content = st.sidebar.selectbox(
    "Tipo de contenido",
    options=content_options,
    index=0
)

source_options_real = sorted([x for x in df["source"].dropna().unique().tolist() if x != ""])
source_options = ["All"] + source_options_real
selected_source = st.sidebar.selectbox(
    "Fuente",
    options=source_options,
    index=0
)

language_options_real = sorted([x for x in df["original_language"].dropna().unique().tolist() if x != ""])
language_options = ["All"] + language_options_real
selected_languages = st.sidebar.multiselect(
    "Idiomas",
    options=language_options,
    default=["All"]
)

available_topics = [TOPIC_LABELS[c] for c in TOPIC_COLS if c in df.columns]
selected_topics = st.sidebar.multiselect(
    "Temas",
    options=available_topics,
    default=[]
)

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider(
        "Rango de años",
        min_value=min_year,
        max_value=max_year,
        value=(min_year, max_year)
    )
else:
    year_range = None

# ──────────────────────────────────────────────────────────────────────────────
# FILTERS
# ──────────────────────────────────────────────────────────────────────────────
filtered_df = df.copy()

if selected_content != "All" and "content_type" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["content_type"] == selected_content]

if selected_source != "All" and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"] == selected_source]

if "All" not in selected_languages and selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]

if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")
    ]

selected_topic_cols = [
    col for col, label in TOPIC_LABELS.items() if label in selected_topics
]
for col in selected_topic_cols:
    if col in filtered_df.columns:
        filtered_df = filtered_df[filtered_df[col] == 1]

st.markdown(f"### Dataset filtrado: {filtered_df.shape[0]:,} títulos")

# ──────────────────────────────────────────────────────────────────────────────
# KPIS
# ──────────────────────────────────────────────────────────────────────────────
total_titles = len(filtered_df)
avg_popularity = filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan
avg_vote = filtered_df["vote_average"].mean() if "vote_average" in filtered_df.columns else np.nan
avg_business = filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total títulos", f"{total_titles:,}")
k2.metric("Popularidad promedio", f"{avg_popularity:.2f}" if pd.notna(avg_popularity) else "N/A")
k3.metric("Rating promedio", f"{avg_vote:.2f}" if pd.notna(avg_vote) else "N/A")
k4.metric("Business Value promedio", f"{avg_business:.2f}" if pd.notna(avg_business) else "N/A")

st.divider()

# ──────────────────────────────────────────────────────────────────────────────
# TABS
# ──────────────────────────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Overview",
    "🎯 Marketing Performance",
    "🔎 Similar Titles Finder",
    "🧠 Synopsis Matcher",
    "🗂 Dataset Explorer"
])

# ══════════════════════════════════════════════════════════════════════════════
# TAB 1 – OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════
with tab1:
    st.subheader("Vista general del catálogo")

    c1, c2 = st.columns(2)

    with c1:
        if "content_type" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["content_type"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_1)
            ax.set_title("Títulos por tipo de contenido", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Cantidad", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "source" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["source"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_2)
            ax.set_title("Títulos por fuente", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Cantidad", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "popularity" in filtered_df.columns and filtered_df["popularity"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["popularity"].dropna(), bins=30, color=ACCENT_3)
            ax.set_title("Distribución de popularidad", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Popularity", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c4:
        if "vote_average" in filtered_df.columns and filtered_df["vote_average"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["vote_average"].dropna(), bins=30, color=ACCENT_1)
            ax.set_title("Distribución de rating", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Vote Average", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 2 – MARKETING PERFORMANCE
# ══════════════════════════════════════════════════════════════════════════════
with tab2:
    st.subheader("Análisis de marketing y performance")

    metric_map = {
        "Visibility": "visibility_score",
        "Engagement": "engagement_score",
        "Business Value": "business_value_score"
    }

    selected_metric_label = st.radio(
        "Métrica destacada:",
        options=list(metric_map.keys()),
        horizontal=True,
        index=2
    )
    highlight_metric = metric_map[selected_metric_label]

    c1, c2 = st.columns(2)

    with c1:
        if highlight_metric in filtered_df.columns and "title" in filtered_df.columns and not filtered_df.empty:
            top_df = filtered_df[["title", highlight_metric]].dropna().sort_values(
                by=highlight_metric, ascending=False
            ).head(10)

            fig, ax = make_dark_fig((8, 5))
            ax.barh(top_df["title"][::-1], top_df[highlight_metric][::-1], color=ACCENT_1)
            ax.set_title(f"Top 10 por {selected_metric_label}", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "marketing_segment" in filtered_df.columns and not filtered_df.empty:
            seg = filtered_df["marketing_segment"].value_counts()
            fig, ax = make_dark_fig((8, 5))
            ax.bar(seg.index, seg.values, color=ACCENT_2)
            ax.set_title("Distribución por marketing segment", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 3 – SIMILAR TITLES FINDER
# ══════════════════════════════════════════════════════════════════════════════
with tab3:
    st.subheader("Buscador de títulos similares")

    available_titles = sorted(filtered_df["title"].dropna().astype(str).unique().tolist())

    if not available_titles:
        st.warning("No hay títulos disponibles con los filtros actuales.")
    else:
        selected_title = st.selectbox(
            "Empieza a escribir y selecciona un título",
            options=available_titles,
            index=None,
            placeholder="Escribe el nombre de una película o serie..."
        )

        if selected_title:
            selected_row = filtered_df[filtered_df["title"] == selected_title].head(1)

            if not selected_row.empty:
                row = selected_row.iloc[0]

                rank_pop, total_pop = get_rank(df, selected_title, "popularity")
                rank_vote, total_vote = get_rank(df, selected_title, "vote_average")
                rank_business, total_business = get_rank(df, selected_title, "business_value_score")

                st.markdown(f"#### {selected_title}")

                info1, info2, info3, info4 = st.columns(4)
                info1.metric("Tipo", row.get("content_type", "N/A"))
                info2.metric("Año", int(row["release_year"]) if pd.notna(row.get("release_year")) else "N/A")
                info3.metric("Rating", f"{row['vote_average']:.2f}" if pd.notna(row.get("vote_average")) else "N/A")
                info4.metric("Business Value", f"{row['business_value_score']:.2f}" if pd.notna(row.get("business_value_score")) else "N/A")

                st.markdown(
                    f"**Géneros:** {row.get('genre_names', 'N/A')}  \n"
                    f"**Idioma:** {row.get('original_language', 'N/A')}  \n"
                    f"**Marketing Segment:** {row.get('marketing_segment', 'N/A')}  \n"
                    f"**Cluster:** {row.get('cluster_label', 'N/A')}"
                )

                active_topics = get_active_topics_from_row(row)
                st.markdown(f"**Temas detectados:** {', '.join(active_topics) if active_topics else 'No detectados'}")

                if row.get("overview"):
                    st.markdown("**Overview**")
                    st.write(row["overview"])

                r1, r2, r3 = st.columns(3)
                r1.metric("Ranking popularidad", f"#{rank_pop}" if rank_pop else "N/A", delta=f"de {total_pop}" if total_pop else None)
                r2.metric("Ranking rating", f"#{rank_vote}" if rank_vote else "N/A", delta=f"de {total_vote}" if total_vote else None)
                r3.metric("Ranking business value", f"#{rank_business}" if rank_business else "N/A", delta=f"de {total_business}" if total_business else None)

                st.divider()

                similar_df = get_similar_titles(df_similarity, sim_matrix, selected_title, top_n=10)

                st.markdown("#### Títulos similares")

                if similar_df.empty:
                    st.info("No se encontraron títulos similares.")
                else:
                    fig, ax = make_dark_fig((9, 5))
                    plot_df = similar_df.head(8).copy()
                    ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_1)
                    ax.set_title("Top similitud", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                    ax.set_xlabel("Similarity Score", color=TEXT)
                    st.pyplot(fig)
                    plt.close(fig)

                    display_df = similar_df.copy()
                    if "similarity_score" in display_df.columns:
                        display_df["similarity_score"] = display_df["similarity_score"].round(3)

                    st.dataframe(display_df, use_container_width=True)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 4 – SYNOPSIS MATCHER
# ══════════════════════════════════════════════════════════════════════════════
with tab4:
    st.subheader("Synopsis Matcher")
    st.markdown(
        "Escribe una sinopsis, elige si es película, serie o ambos, "
        "y te mostramos los temas detectados y los títulos más parecidos del dataset."
    )

    user_type = st.selectbox(
        "Tipo de contenido",
        options=["All", "movie", "tv"],
        index=0
    )

    user_synopsis = st.text_area(
        "Escribe la sinopsis",
        height=180,
        placeholder="Ejemplo: Una joven periodista descubre una red de corrupción política mientras intenta salvar su carrera y proteger a su familia..."
    )

    top_n = st.slider("Número de comparables", min_value=3, max_value=10, value=3)

    if st.button("Analizar sinopsis"):
        if not user_synopsis or not user_synopsis.strip():
            st.warning("Por favor, escribe una sinopsis.")
        else:
            detected_topics = detect_topics_from_synopsis(user_synopsis)

            st.markdown("### Temas detectados")
            st.write(", ".join(detected_topics))

            similar_titles = get_top_similar_titles(
                user_synopsis=user_synopsis,
                content_type=user_type,
                df=filtered_df,
                top_n=top_n
            )

            st.markdown(f"### Top {top_n} títulos más parecidos")

            if similar_titles.empty:
                st.info("No se encontraron títulos comparables con esos filtros.")
            else:
                for i, (_, row) in enumerate(similar_titles.iterrows(), start=1):
                    st.markdown(f"#### #{i} - {row.get('title', 'Unknown title')}")
                    st.write(f"**Tipo:** {row.get('content_type', 'N/A')}")
                    st.write(f"**Géneros:** {row.get('genre_names', 'N/A')}")
                    st.write(f"**Idioma:** {row.get('original_language', 'N/A')}")
                    st.write(f"**Año:** {row.get('release_year', 'N/A')}")
                    st.write(f"**Text similarity:** {row.get('text_similarity', 0):.3f}")
                    st.write(f"**Topic similarity:** {row.get('topic_similarity', 0):.3f}")
                    st.write(f"**Similarity score:** {row.get('similarity_score', 0):.3f}")

                    if "business_value_score" in row and pd.notna(row["business_value_score"]):
                        st.write(f"**Business Value Score:** {row['business_value_score']:.2f}")

                    if "predicted_business_value" in row and pd.notna(row["predicted_business_value"]):
                        st.write(f"**Predicted Business Value:** {row['predicted_business_value']:.2f}")

                    st.write(f"**Overview:** {row.get('overview', 'N/A')}")
                    st.markdown("---")

                avg_bv = similar_titles["business_value_score"].mean() if "business_value_score" in similar_titles.columns else np.nan

                st.markdown("### Lectura rápida")
                if pd.notna(avg_bv):
                    if avg_bv >= 70:
                        st.success("La idea se parece a contenidos con alto potencial comercial.")
                    elif avg_bv >= 45:
                        st.info("La idea se parece a contenidos con potencial medio; podría depender del marketing y posicionamiento.")
                    else:
                        st.warning("La idea se parece a contenidos con menor valor comercial estimado en el dataset.")

# ══════════════════════════════════════════════════════════════════════════════
# TAB 5 – DATASET EXPLORER
# ══════════════════════════════════════════════════════════════════════════════
with tab5:
    st.subheader("Explorador del dataset")

    explorer_cols = [c for c in [
        "title", "content_type", "genre_names", "release_year",
        "original_language", "source", "popularity", "vote_average",
        "vote_count", "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value",
        "marketing_segment", "cluster_label", "overview"
    ] if c in filtered_df.columns]

    st.dataframe(filtered_df[explorer_cols].head(100), use_container_width=True)

    csv = filtered_df.to_csv(index=False).encode("utf-8-sig")
    st.download_button(
        label="Descargar dataset filtrado",
        data=csv,
        file_name="filtered_streaming_dataset.csv",
        mime="text/csv"
    )
'''

app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    webbrowser.open(url)
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()

Starting Streamlit on http://127.0.0.1:55483 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:55483

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:55483


2026-03-24 17:39:46.385 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:40:14.001 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:40:39.740 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:40:39.762 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:40:54.918 

In [8]:
import socket
import subprocess
import time
import urllib.request
import sys
import webbrowser
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False

streamlit_code = r'''
import re
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Streaming Marketing Analytics",
    page_icon="🎬",
    layout="wide"
)

CARD_BG = "#1E293B"
FIG_BG = "#0F172A"
TEXT = "white"
MUTED = "#94A3B8"
ACCENT_1 = "#6366F1"
ACCENT_2 = "#10B981"
ACCENT_3 = "#F59E0B"
ACCENT_4 = "#EF4444"

TOPIC_COLS = [
    "topic_lgbtq",
    "topic_politics",
    "topic_climate",
    "topic_war",
    "topic_family",
    "topic_crime",
    "topic_romance",
    "topic_technology",
]

TOPIC_LABELS = {
    "topic_lgbtq": "LGBTQ+",
    "topic_politics": "Politics",
    "topic_climate": "Climate",
    "topic_war": "War",
    "topic_family": "Family",
    "topic_crime": "Crime",
    "topic_romance": "Romance",
    "topic_technology": "Technology",
}

TOPIC_KEYWORDS = {
    "LGBTQ+": ["gay", "lesbian", "lgbt", "trans", "queer", "bisexual", "drag"],
    "Politics": ["politics", "president", "government", "election", "policy", "minister", "senate", "congress"],
    "Climate": ["climate", "environment", "global warming", "pollution", "sustainability", "ecology"],
    "War": ["war", "battle", "army", "soldier", "military", "conflict", "invasion"],
    "Family": ["family", "mother", "father", "children", "home", "parent", "siblings"],
    "Crime": ["crime", "murder", "police", "detective", "investigation", "gang", "mafia", "robbery"],
    "Romance": ["love", "romance", "relationship", "couple", "marriage", "affair"],
    "Technology": ["technology", "ai", "robot", "future", "cyber", "machine", "algorithm"],
    "Mental Health": ["depression", "anxiety", "trauma", "therapy", "mental", "psychological"],
    "Coming of Age": ["teen", "adolescent", "growing up", "school", "youth", "friendship"],
    "Social Issues": ["racism", "inequality", "poverty", "discrimination", "justice", "migration"],
    "Fantasy / Supernatural": ["magic", "witch", "dragon", "supernatural", "monster", "curse", "fantasy"],
}


# ──────────────────────────────────────────────────────────────────────────────
# STYLE
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
.block-container {
    padding-top: 1.4rem;
    padding-bottom: 2rem;
}
div[data-testid="stMetric"] {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    padding: 14px 16px;
    border-radius: 16px;
}
div[data-testid="stDataFrame"] {
    border-radius: 12px;
}
.small-card {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 14px;
    padding: 12px 14px;
    margin-top: 8px;
    margin-bottom: 8px;
}
</style>
""", unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9áéíóúñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def detect_topics_from_synopsis(text):
    text_clean = clean_text(text)
    found_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword.lower() in text_clean for keyword in keywords):
            found_topics.append(topic)

    if not found_topics:
        found_topics.append("General / No clear topic")

    return found_topics


def get_active_topics_from_row(row):
    return [TOPIC_LABELS[col] for col in TOPIC_COLS if row.get(col, 0) == 1]


def make_dark_fig(figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(FIG_BG)
    ax.set_facecolor(CARD_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT)
    return fig, ax


def safe_float(value):
    return float(value) if pd.notna(value) else np.nan


def metric_str(value, decimals=2):
    return f"{value:.{decimals}f}" if pd.notna(value) else "N/A"


# ──────────────────────────────────────────────────────────────────────────────
# DATA
# ──────────────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("final_streaming_dataset.csv")

    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    if "release_year" in df.columns:
        df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

    numeric_cols = [
        "popularity", "vote_average", "vote_count",
        "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value",
        "topic_diversity_score", "runtime_final", "cluster"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    text_cols = [
        "title", "overview", "overview_clean", "genre_names",
        "content_type", "source", "original_language",
        "marketing_segment", "cluster_label", "production_companies"
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    for col in TOPIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        else:
            df[col] = 0

    df["title_key"] = df["title"].fillna("").astype(str).str.lower().str.strip()

    if "overview_clean" not in df.columns:
        df["overview_clean"] = df["overview"].fillna("").apply(clean_text)
    else:
        df["overview_clean"] = df["overview_clean"].fillna("").apply(clean_text)

    return df


@st.cache_data
def prepare_similarity(df):
    work_df = df.copy()

    work_df["topics_text"] = work_df.apply(
        lambda row: " ".join(get_active_topics_from_row(row)),
        axis=1
    )

    work_df["similarity_text"] = (
        work_df["title"].fillna("") + " "
        + work_df["genre_names"].fillna("").str.replace(",", " ", regex=False) + " "
        + work_df["overview_clean"].fillna("") + " "
        + work_df["overview"].fillna("") + " "
        + work_df["topics_text"].fillna("")
    ).str.lower()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )
    matrix = vectorizer.fit_transform(work_df["similarity_text"])

    return work_df, matrix


@st.cache_resource
def build_similarity_model(df):
    model_df = df.copy()

    if "overview" not in model_df.columns:
        model_df["overview"] = ""

    model_df["overview_clean"] = model_df["overview"].fillna("").apply(clean_text)

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform(model_df["overview_clean"])

    return vectorizer, tfidf_matrix, model_df


def get_similar_titles(df_similarity, matrix, selected_title, top_n=10):
    matches = df_similarity.index[df_similarity["title"] == selected_title].tolist()
    if not matches:
        return pd.DataFrame()

    idx = matches[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()

    sim_df = df_similarity.copy()
    sim_df["similarity_score"] = sims
    sim_df = sim_df[sim_df.index != idx].copy()

    rank_cols = [
        "title", "content_type", "genre_names", "release_year",
        "vote_average", "popularity", "visibility_score",
        "engagement_score", "business_value_score",
        "predicted_business_value", "marketing_segment",
        "cluster_label", "similarity_score"
    ]
    existing_cols = [c for c in rank_cols if c in sim_df.columns]

    return sim_df.sort_values(
        by=["similarity_score", "business_value_score", "vote_average", "popularity"],
        ascending=[False, False, False, False]
    )[existing_cols].head(top_n)


def get_top_similar_titles(user_synopsis, content_type, df, top_n=3):
    vectorizer, tfidf_matrix, model_df = build_similarity_model(df)

    filtered = model_df.copy()

    if "content_type" in filtered.columns and content_type and content_type.lower() != "all":
        filtered = filtered[
            filtered["content_type"].astype(str).str.lower() == content_type.lower()
        ].copy()

    if filtered.empty:
        return pd.DataFrame()

    filtered_indices = filtered.index.tolist()

    user_text = clean_text(user_synopsis)
    user_vec = vectorizer.transform([user_text])

    similarities = cosine_similarity(user_vec, tfidf_matrix[filtered_indices]).flatten()
    filtered["text_similarity"] = similarities

    user_topics = set(detect_topics_from_synopsis(user_synopsis))

    if "overview" in filtered.columns:
        filtered["detected_topics_temp"] = filtered["overview"].fillna("").apply(detect_topics_from_synopsis)

        def topic_overlap_score(topics_list):
            item_topics = set(topics_list)
            if not user_topics or not item_topics:
                return 0
            return len(user_topics.intersection(item_topics)) / max(len(user_topics), 1)

        filtered["topic_similarity"] = filtered["detected_topics_temp"].apply(topic_overlap_score)
    else:
        filtered["topic_similarity"] = 0

    filtered["similarity_score"] = (
        0.8 * filtered["text_similarity"] +
        0.2 * filtered["topic_similarity"]
    )

    cols_to_show = [
        col for col in [
            "title",
            "content_type",
            "genre_names",
            "original_language",
            "release_year",
            "overview",
            "business_value_score",
            "predicted_business_value",
            "text_similarity",
            "topic_similarity",
            "similarity_score"
        ] if col in filtered.columns
    ]

    result = filtered.sort_values("similarity_score", ascending=False).head(top_n)
    return result[cols_to_show]


def get_rank(df_all, title, col):
    if col not in df_all.columns:
        return None, None

    rank_df = df_all[["title", col]].dropna().sort_values(by=col, ascending=False).reset_index(drop=True)
    rank_df.index = rank_df.index + 1
    matches = rank_df[rank_df["title"] == title]

    if matches.empty:
        return None, len(rank_df)

    return int(matches.index[0]), len(rank_df)


# ──────────────────────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────────────────────
df = load_data()
df_similarity, sim_matrix = prepare_similarity(df)

# ──────────────────────────────────────────────────────────────────────────────
# HEADER
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("### 🎬 Streaming Marketing Analytics")
st.markdown("**Dashboard de visibilidad, engagement, valor comercial y similitud entre títulos**")
st.divider()

# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────────────────────────────────────
st.sidebar.header("Filtros")

content_options_real = sorted([x for x in df["content_type"].dropna().unique().tolist() if x != ""])
content_options = ["All"] + content_options_real
selected_content = st.sidebar.selectbox(
    "Tipo de contenido",
    options=content_options,
    index=0
)

source_options_real = sorted([x for x in df["source"].dropna().unique().tolist() if x != ""])
source_options = ["All"] + source_options_real
selected_source = st.sidebar.selectbox(
    "Fuente",
    options=source_options,
    index=0
)

language_options_real = sorted([x for x in df["original_language"].dropna().unique().tolist() if x != ""])
language_options = ["All"] + language_options_real
selected_languages = st.sidebar.multiselect(
    "Idiomas",
    options=language_options,
    default=["All"]
)

available_topics = [TOPIC_LABELS[c] for c in TOPIC_COLS if c in df.columns]
selected_topics = st.sidebar.multiselect(
    "Temas",
    options=available_topics,
    default=[]
)

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider(
        "Rango de años",
        min_value=min_year,
        max_value=max_year,
        value=(min_year, max_year)
    )
else:
    year_range = None

st.sidebar.divider()
st.sidebar.markdown("### 🔎 Title Quick Search")

sidebar_title_options = sorted(df["title"].dropna().astype(str).unique().tolist())
sidebar_selected_title = st.sidebar.selectbox(
    "Busca un título",
    options=sidebar_title_options,
    index=None,
    placeholder="Empieza a escribir un título..."
)

# ──────────────────────────────────────────────────────────────────────────────
# FILTERS
# ──────────────────────────────────────────────────────────────────────────────
filtered_df = df.copy()

if selected_content != "All" and "content_type" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["content_type"] == selected_content]

if selected_source != "All" and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"] == selected_source]

if "All" not in selected_languages and selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]

if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")
    ]

selected_topic_cols = [
    col for col, label in TOPIC_LABELS.items() if label in selected_topics
]
for col in selected_topic_cols:
    if col in filtered_df.columns:
        filtered_df = filtered_df[filtered_df[col] == 1]

# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR QUICK TITLE RESULT
# ──────────────────────────────────────────────────────────────────────────────
if sidebar_selected_title:
    sidebar_row = df[df["title"] == sidebar_selected_title].head(1)

    if not sidebar_row.empty:
        row = sidebar_row.iloc[0]

        st.sidebar.markdown("#### Información del título")
        st.sidebar.markdown(
            f"**Tipo:** {row.get('content_type', 'N/A')}  \n"
            f"**Año:** {int(row['release_year']) if pd.notna(row.get('release_year')) else 'N/A'}  \n"
            f"**Idioma:** {row.get('original_language', 'N/A')}"
        )

        st.sidebar.markdown("#### Overview")
        overview_text = row.get("overview", "")
        if overview_text:
            st.sidebar.caption(overview_text[:350] + ("..." if len(overview_text) > 350 else ""))
        else:
            st.sidebar.caption("No disponible.")

        st.sidebar.markdown("#### Marketing performance")
        st.sidebar.metric("Visibility", metric_str(row.get("visibility_score", np.nan)))
        st.sidebar.metric("Engagement", metric_str(row.get("engagement_score", np.nan)))
        st.sidebar.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))
        if "predicted_business_value" in row.index:
            st.sidebar.metric("Predicted BV", metric_str(row.get("predicted_business_value", np.nan)))

st.markdown(f"### Dataset filtrado: {filtered_df.shape[0]:,} títulos")

# ──────────────────────────────────────────────────────────────────────────────
# KPIS
# ──────────────────────────────────────────────────────────────────────────────
total_titles = len(filtered_df)
avg_popularity = filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan
avg_vote = filtered_df["vote_average"].mean() if "vote_average" in filtered_df.columns else np.nan
avg_business = filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total títulos", f"{total_titles:,}")
k2.metric("Popularidad promedio", f"{avg_popularity:.2f}" if pd.notna(avg_popularity) else "N/A")
k3.metric("Rating promedio", f"{avg_vote:.2f}" if pd.notna(avg_vote) else "N/A")
k4.metric("Business Value promedio", f"{avg_business:.2f}" if pd.notna(avg_business) else "N/A")

st.divider()

# ──────────────────────────────────────────────────────────────────────────────
# TABS
# ──────────────────────────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Overview",
    "🎯 Marketing Performance",
    "🔎 Similar Titles Finder",
    "🧠 Synopsis Matcher",
    "🗂 Dataset Explorer"
])

# ══════════════════════════════════════════════════════════════════════════════
# TAB 1 – OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════
with tab1:
    st.subheader("Vista general del catálogo")

    c1, c2 = st.columns(2)

    with c1:
        if "content_type" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["content_type"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_1)
            ax.set_title("Títulos por tipo de contenido", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Cantidad", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "source" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["source"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_2)
            ax.set_title("Títulos por fuente", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Cantidad", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "popularity" in filtered_df.columns and filtered_df["popularity"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["popularity"].dropna(), bins=30, color=ACCENT_3)
            ax.set_title("Distribución de popularidad", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Popularity", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c4:
        if "vote_average" in filtered_df.columns and filtered_df["vote_average"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["vote_average"].dropna(), bins=30, color=ACCENT_1)
            ax.set_title("Distribución de rating", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Vote Average", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 2 – MARKETING PERFORMANCE
# ══════════════════════════════════════════════════════════════════════════════
with tab2:
    st.subheader("Análisis de marketing y performance")

    metric_map = {
        "Visibility": "visibility_score",
        "Engagement": "engagement_score",
        "Business Value": "business_value_score"
    }

    selected_metric_label = st.radio(
        "Métrica destacada:",
        options=list(metric_map.keys()),
        horizontal=True,
        index=2
    )
    highlight_metric = metric_map[selected_metric_label]

    c1, c2 = st.columns(2)

    with c1:
        if highlight_metric in filtered_df.columns and "title" in filtered_df.columns and not filtered_df.empty:
            top_df = filtered_df[["title", highlight_metric]].dropna().sort_values(
                by=highlight_metric, ascending=False
            ).head(10)

            fig, ax = make_dark_fig((8, 5))
            ax.barh(top_df["title"][::-1], top_df[highlight_metric][::-1], color=ACCENT_1)
            ax.set_title(f"Top 10 por {selected_metric_label}", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "marketing_segment" in filtered_df.columns and not filtered_df.empty:
            seg = filtered_df["marketing_segment"].value_counts()
            fig, ax = make_dark_fig((8, 5))
            ax.bar(seg.index, seg.values, color=ACCENT_2)
            ax.set_title("Distribución por marketing segment", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 3 – SIMILAR TITLES FINDER
# ══════════════════════════════════════════════════════════════════════════════
with tab3:
    st.subheader("Buscador de títulos similares")

    available_titles = sorted(filtered_df["title"].dropna().astype(str).unique().tolist())

    if not available_titles:
        st.warning("No hay títulos disponibles con los filtros actuales.")
    else:
        selected_title = st.selectbox(
            "Empieza a escribir y selecciona un título",
            options=available_titles,
            index=None,
            placeholder="Escribe el nombre de una película o serie..."
        )

        similar_df = pd.DataFrame()

        if selected_title:
            selected_row = filtered_df[filtered_df["title"] == selected_title].head(1)

            if not selected_row.empty:
                row = selected_row.iloc[0]

                rank_pop, total_pop = get_rank(df, selected_title, "popularity")
                rank_vote, total_vote = get_rank(df, selected_title, "vote_average")
                rank_business, total_business = get_rank(df, selected_title, "business_value_score")

                st.markdown(f"#### {selected_title}")

                info1, info2, info3, info4 = st.columns(4)
                info1.metric("Tipo", row.get("content_type", "N/A"))
                info2.metric("Año", int(row["release_year"]) if pd.notna(row.get("release_year")) else "N/A")
                info3.metric("Rating", f"{row['vote_average']:.2f}" if pd.notna(row.get("vote_average")) else "N/A")
                info4.metric("Business Value", f"{row['business_value_score']:.2f}" if pd.notna(row.get("business_value_score")) else "N/A")

                st.markdown(
                    f"**Géneros:** {row.get('genre_names', 'N/A')}  \n"
                    f"**Idioma:** {row.get('original_language', 'N/A')}  \n"
                    f"**Marketing Segment:** {row.get('marketing_segment', 'N/A')}  \n"
                    f"**Cluster:** {row.get('cluster_label', 'N/A')}"
                )

                active_topics = get_active_topics_from_row(row)
                st.markdown(f"**Temas detectados:** {', '.join(active_topics) if active_topics else 'No detectados'}")

                if row.get("overview"):
                    st.markdown("**Overview**")
                    st.write(row["overview"])

                r1, r2, r3 = st.columns(3)
                r1.metric("Ranking popularidad", f"#{rank_pop}" if rank_pop else "N/A", delta=f"de {total_pop}" if total_pop else None)
                r2.metric("Ranking rating", f"#{rank_vote}" if rank_vote else "N/A", delta=f"de {total_vote}" if total_vote else None)
                r3.metric("Ranking business value", f"#{rank_business}" if rank_business else "N/A", delta=f"de {total_business}" if total_business else None)

                st.divider()

                similar_df = get_similar_titles(df_similarity, sim_matrix, selected_title, top_n=10)

                st.markdown("#### Títulos similares")

                if similar_df.empty:
                    st.info("No se encontraron títulos similares.")
                else:
                    fig, ax = make_dark_fig((9, 5))
                    plot_df = similar_df.head(8).copy()
                    ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_1)
                    ax.set_title("Top similitud", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                    ax.set_xlabel("Similarity Score", color=TEXT)
                    st.pyplot(fig)
                    plt.close(fig)

                    display_df = similar_df.copy()
                    if "similarity_score" in display_df.columns:
                        display_df["similarity_score"] = display_df["similarity_score"].round(3)

                    st.dataframe(display_df, use_container_width=True)

        if not similar_df.empty:
            download_similar_df = similar_df.copy()
            if "similarity_score" in download_similar_df.columns:
                download_similar_df["similarity_score"] = download_similar_df["similarity_score"].round(5)

            csv_similar = download_similar_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button(
                label="Descargar títulos similares",
                data=csv_similar,
                file_name="similar_titles_results.csv",
                mime="text/csv"
            )

# ══════════════════════════════════════════════════════════════════════════════
# TAB 4 – SYNOPSIS MATCHER
# ══════════════════════════════════════════════════════════════════════════════
with tab4:
    st.subheader("Synopsis Matcher")
    st.markdown(
        "Escribe una sinopsis, elige si es película, serie o ambos, "
        "y te mostramos los temas detectados y los títulos más parecidos del dataset."
    )

    user_type = st.selectbox(
        "Tipo de contenido",
        options=["All", "movie", "tv"],
        index=0
    )

    user_synopsis = st.text_area(
        "Escribe la sinopsis",
        height=180,
        placeholder="Ejemplo: Una joven periodista descubre una red de corrupción política mientras intenta salvar su carrera y proteger a su familia..."
    )

    top_n = st.slider("Número de comparables", min_value=3, max_value=10, value=3)

    synopsis_results = pd.DataFrame()
    detected_topics = []

    if st.button("Analizar sinopsis"):
        if not user_synopsis or not user_synopsis.strip():
            st.warning("Por favor, escribe una sinopsis.")
        else:
            detected_topics = detect_topics_from_synopsis(user_synopsis)

            st.markdown("### Temas detectados")
            st.write(", ".join(detected_topics))

            synopsis_results = get_top_similar_titles(
                user_synopsis=user_synopsis,
                content_type=user_type,
                df=filtered_df,
                top_n=top_n
            )

            st.session_state["synopsis_results"] = synopsis_results
            st.session_state["user_synopsis_text"] = user_synopsis
            st.session_state["user_synopsis_topics"] = detected_topics
            st.session_state["user_synopsis_type"] = user_type

    if "synopsis_results" in st.session_state:
        synopsis_results = st.session_state["synopsis_results"]
        user_synopsis = st.session_state.get("user_synopsis_text", "")
        detected_topics = st.session_state.get("user_synopsis_topics", [])
        user_type_saved = st.session_state.get("user_synopsis_type", "All")

        if detected_topics:
            st.markdown("### Temas detectados")
            st.write(", ".join(detected_topics))

        st.markdown(f"### Top {len(synopsis_results)} títulos más parecidos")

        if synopsis_results.empty:
            st.info("No se encontraron títulos comparables con esos filtros.")
        else:
            for i, (_, row) in enumerate(synopsis_results.iterrows(), start=1):
                st.markdown(f"#### #{i} - {row.get('title', 'Unknown title')}")
                st.write(f"**Tipo:** {row.get('content_type', 'N/A')}")
                st.write(f"**Géneros:** {row.get('genre_names', 'N/A')}")
                st.write(f"**Idioma:** {row.get('original_language', 'N/A')}")
                st.write(f"**Año:** {row.get('release_year', 'N/A')}")
                st.write(f"**Text similarity:** {row.get('text_similarity', 0):.3f}")
                st.write(f"**Topic similarity:** {row.get('topic_similarity', 0):.3f}")
                st.write(f"**Similarity score:** {row.get('similarity_score', 0):.3f}")

                if "business_value_score" in row and pd.notna(row["business_value_score"]):
                    st.write(f"**Business Value Score:** {row['business_value_score']:.2f}")

                if "predicted_business_value" in row and pd.notna(row["predicted_business_value"]):
                    st.write(f"**Predicted Business Value:** {row['predicted_business_value']:.2f}")

                st.write(f"**Overview:** {row.get('overview', 'N/A')}")
                st.markdown("---")

            avg_bv = synopsis_results["business_value_score"].mean() if "business_value_score" in synopsis_results.columns else np.nan

            st.markdown("### Lectura rápida")
            if pd.notna(avg_bv):
                if avg_bv >= 70:
                    st.success("La idea se parece a contenidos con alto potencial comercial.")
                elif avg_bv >= 45:
                    st.info("La idea se parece a contenidos con potencial medio; podría depender del marketing y posicionamiento.")
                else:
                    st.warning("La idea se parece a contenidos con menor valor comercial estimado en el dataset.")

            export_df = synopsis_results.copy()
            export_df.insert(0, "input_synopsis", user_synopsis)
            export_df.insert(1, "input_content_type", user_type_saved)
            export_df.insert(2, "detected_topics", ", ".join(detected_topics))

            csv_synopsis = export_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button(
                label="Descargar sinopsis + títulos similares",
                data=csv_synopsis,
                file_name="synopsis_matcher_results.csv",
                mime="text/csv"
            )

# ══════════════════════════════════════════════════════════════════════════════
# TAB 5 – DATASET EXPLORER
# ══════════════════════════════════════════════════════════════════════════════
with tab5:
    st.subheader("Explorador del dataset")

    explorer_cols = [c for c in [
        "title", "content_type", "genre_names", "release_year",
        "original_language", "source", "popularity", "vote_average",
        "vote_count", "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value",
        "marketing_segment", "cluster_label", "overview"
    ] if c in filtered_df.columns]

    st.dataframe(filtered_df[explorer_cols].head(100), use_container_width=True)

    csv = filtered_df.to_csv(index=False).encode("utf-8-sig")
    st.download_button(
        label="Descargar dataset filtrado",
        data=csv,
        file_name="filtered_streaming_dataset.csv",
        mime="text/csv"
    )
'''

app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    webbrowser.open(url)
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()

Starting Streamlit on http://127.0.0.1:57481 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:57481

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:57481


2026-03-24 17:55:38.684 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:56:16.524 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:56:16.546 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:56:55.794 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 17:56:55.811 

(personal note ): this might be close to a final version 

In [ ]:
import socket
import subprocess
import time
import urllib.request
import sys
import webbrowser
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False

streamlit_code = r'''
import re
import ast
from pathlib import Path

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Streaming Marketing Intelligence",
    page_icon="🎬",
    layout="wide"
)

CARD_BG = "#1E293B"
FIG_BG = "#0F172A"
TEXT = "white"
MUTED = "#94A3B8"
ACCENT_1 = "#6366F1"
ACCENT_2 = "#10B981"
ACCENT_3 = "#F59E0B"
ACCENT_4 = "#EF4444"
ACCENT_5 = "#22C55E"

DEFAULT_DATASET_CANDIDATES = [
    
    "DATA/PROCESSED/all_streaming_titles.csv",
]

TOPIC_KEYWORDS = {
    "lgbtq": [
        "gay", "lesbian", "lgbt", "trans", "queer", "bisexual",
        "nonbinary", "coming out", "drag", "identity"
    ],

    "politics": [
        "politics", "president", "government", "election", "policy",
        "senate", "congress", "campaign", "minister", "dictator",
        "democracy", "corruption", "power", "state"
    ],

    "climate_environment": [
        "climate", "environment", "global warming", "pollution",
        "ecology", "sustainability", "nature", "forest", "wildlife",
        "environmental disaster", "carbon", "climate crisis"
    ],

    "war_military": [
        "war", "battle", "army", "soldier", "military", "conflict",
        "weapon", "navy", "air force", "commander", "resistance",
        "invasion", "combat", "veteran"
    ],

    "family": [
        "family", "mother", "father", "parent", "children", "child",
        "home", "siblings", "brother", "sister", "marriage",
        "divorce", "parenthood", "relatives"
    ],

    "crime": [
        "crime", "murder", "police", "detective", "investigation",
        "killer", "gang", "mafia", "cartel", "robbery", "heist",
        "forensics", "prison", "criminal", "underworld"
    ],

    "policial": [
        "police", "cop", "officer", "detective", "agent", "rookie",
        "investigation", "investigator", "inspector", "sergeant",
        "lieutenant", "captain", "chief", "precinct", "partner",
        "case", "crime scene", "forensics", "suspect", "witness",
        "interrogation", "undercover", "surveillance", "homicide",
        "manhunt", "law enforcement", "special unit", "task force",
        "fbi", "cia", "narcotics", "patrol", "unit"
    ],

    "romance": [
        "love", "romance", "relationship", "couple", "passion",
        "heartbreak", "dating", "affair", "wedding", "breakup",
        "soulmate", "jealousy"
    ],

    "technology": [
        "technology", "ai", "artificial intelligence", "robot",
        "future", "cyber", "computer", "hacker", "internet",
        "virtual reality", "machine", "automation", "surveillance"
    ],

    "mental_health": [
        "depression", "anxiety", "trauma", "therapy", "mental",
        "psychological", "stress", "grief", "addiction", "bipolar",
        "schizophrenia", "panic", "healing", "psychiatric"
    ],

    "coming_of_age": [
        "teen", "adolescent", "growing up", "school", "youth",
        "friendship", "identity", "first love", "high school",
        "college", "self-discovery", "maturity"
    ],

    "social_issues": [
        "racism", "inequality", "poverty", "discrimination", "justice",
        "migration", "sexism", "class", "homophobia", "xenophobia",
        "oppression", "human rights", "activism"
    ],

    "fantasy_supernatural": [
        "magic", "witch", "dragon", "supernatural", "monster",
        "curse", "fantasy", "sorcery", "wizard", "demon", "ghost",
        "prophecy", "kingdom", "spell"
    ],

    "science_fiction": [
        "space", "alien", "spaceship", "future", "planet",
        "time travel", "parallel universe", "android", "mutation",
        "dystopia", "utopia", "interstellar", "extraterrestrial"
    ],

    "horror": [
        "horror", "fear", "haunted", "ghost", "possession",
        "slasher", "evil", "demon", "nightmare", "blood",
        "terror", "zombie", "paranormal"
    ],

    "thriller": [
        "thriller", "suspense", "mystery", "conspiracy", "chase",
        "secret", "obsession", "danger", "kidnapping", "betrayal",
        "tension", "survival"
    ],

    "action_adventure": [
        "action", "adventure", "hero", "mission", "explosion",
        "fight", "chase", "survival", "escape", "journey",
        "quest", "mercenary"
    ],

    "historical": [
        "history", "historical", "period drama", "king", "queen",
        "empire", "revolution", "civilization", "medieval",
        "ancient", "biographical", "royalty"
    ],

    "biography": [
        "biography", "biopic", "true story", "real life", "famous",
        "artist", "scientist", "politician", "athlete", "inventor"
    ],

    "sports": [
        "sport", "football", "soccer", "basketball", "baseball",
        "tennis", "boxing", "fighter", "competition", "tournament",
        "coach", "team", "championship"
    ],

    "music_performance": [
        "music", "band", "singer", "concert", "performance",
        "musician", "song", "dance", "ballet", "opera",
        "stage", "show business"
    ],

    "comedy": [
        "comedy", "funny", "humor", "satire", "parody",
        "awkward", "absurd", "joke", "laugh", "misunderstanding"
    ],

    "drama": [
        "drama", "emotional", "conflict", "sacrifice", "loss",
        "betrayal", "redemption", "personal struggle", "intense"
    ],

    "mystery": [
        "mystery", "secret", "clue", "disappearance", "puzzle",
        "unknown", "hidden truth", "unsolved", "investigation"
    ],

    "survival_disaster": [
        "survival", "disaster", "earthquake", "tsunami", "fire",
        "shipwreck", "plane crash", "apocalypse", "epidemic",
        "outbreak", "catastrophe"
    ],

    "religion_spirituality": [
        "religion", "faith", "god", "church", "priest", "spiritual",
        "belief", "miracle", "sacred", "ritual", "afterlife"
    ],

    "animation_family": [
        "animation", "animated", "family-friendly", "kids",
        "talking animals", "fairy tale", "adventure for children"
    ],

    "friendship": [
        "friendship", "best friends", "companionship", "bond",
        "loyalty", "group of friends", "reunion"
    ],

    "revenge": [
        "revenge", "vengeance", "payback", "betrayal", "justice",
        "retaliation", "avenger"
    ],

    "road_trip_journey": [
        "road trip", "journey", "travel", "on the road", "escape",
        "self-discovery", "destination"
    ],

    "school_university": [
        "school", "teacher", "student", "classroom", "university",
        "college", "campus", "exam", "bullying", "graduation"
    ],

    "work_business": [
        "work", "office", "career", "boss", "company", "business",
        "corporate", "startup", "ambition", "promotion", "colleague"
    ]
}


# ──────────────────────────────────────────────────────────────────────────────
# STYLE
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
.block-container {
    padding-top: 1.2rem;
    padding-bottom: 2rem;
}
div[data-testid="stMetric"] {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    padding: 14px 16px;
    border-radius: 16px;
}
div[data-testid="stDataFrame"] {
    border-radius: 12px;
}
.small-card {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 14px;
    padding: 12px 14px;
    margin-top: 8px;
    margin-bottom: 8px;
}
.objective-card {
    background: linear-gradient(135deg, #0F172A 0%, #111827 100%);
    border: 1px solid #1E293B;
    border-radius: 18px;
    padding: 18px 20px;
    margin-bottom: 16px;
}
.section-title {
    font-size: 1.1rem;
    font-weight: 700;
    color: white;
    margin-bottom: 0.5rem;
}
.muted {
    color: #94A3B8;
}
</style>
""", unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9áéíóúñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def metric_str(value, decimals=2):
    return f"{value:.{decimals}f}" if pd.notna(value) else "N/A"


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def parse_genres(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [g.strip() for g in text.split(",") if g.strip()]


def detect_topics_from_synopsis(text: str):
    text_clean = clean_text(text)
    found_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword.lower() in text_clean for keyword in keywords):
            found_topics.append(topic)

    if not found_topics:
        found_topics.append("General / No clear topic")

    return found_topics


def get_active_topics_from_row(row):
    topics = []

    if "top_topics" in row.index and pd.notna(row.get("top_topics")) and str(row.get("top_topics")).strip():
        raw = str(row.get("top_topics"))
        if "|" in raw:
            topics.extend([x.strip() for x in raw.split("|") if x.strip()])
        elif "," in raw:
            topics.extend([x.strip() for x in raw.split(",") if x.strip()])
        else:
            topics.append(raw.strip())

    topic_like_cols = [c for c in row.index if c.startswith("topic_") and c != "topic_diversity_score"]
    for col in topic_like_cols:
        try:
            if float(row.get(col, 0)) == 1:
                topics.append(col.replace("topic_", "").replace("_", " ").title())
        except Exception:
            continue

    topics = [t for t in topics if t]
    return sorted(list(set(topics)))


def make_dark_fig(figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(FIG_BG)
    ax.set_facecolor(CARD_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT)
    return fig, ax


def normalize_0_100(series):
    series = pd.to_numeric(series, errors="coerce")
    valid = series.dropna()
    if valid.empty:
        return pd.Series(np.nan, index=series.index)
    min_v = valid.min()
    max_v = valid.max()
    if min_v == max_v:
        return pd.Series(50.0, index=series.index)
    return ((series - min_v) / (max_v - min_v)) * 100


def ensure_business_value_score(df):
    if "business_value_score" in df.columns and df["business_value_score"].notna().any():
        return df

    pop_norm = normalize_0_100(df["popularity"]) if "popularity" in df.columns else pd.Series(np.nan, index=df.index)
    vote_avg_norm = normalize_0_100(df["vote_average"]) if "vote_average" in df.columns else pd.Series(np.nan, index=df.index)
    vote_count_norm = normalize_0_100(df["vote_count"]) if "vote_count" in df.columns else pd.Series(np.nan, index=df.index)
    visibility = normalize_0_100(df["visibility_score"]) if "visibility_score" in df.columns else pd.Series(np.nan, index=df.index)
    engagement = normalize_0_100(df["engagement_score"]) if "engagement_score" in df.columns else pd.Series(np.nan, index=df.index)
    reception = normalize_0_100(df["audience_reception_score"]) if "audience_reception_score" in df.columns else pd.Series(np.nan, index=df.index)

    pieces = pd.concat(
        [pop_norm, vote_avg_norm, vote_count_norm, visibility, engagement, reception],
        axis=1
    )
    pieces.columns = ["pop", "vote_avg", "vote_count", "visibility", "engagement", "reception"]

    weights = {
        "pop": 0.25,
        "vote_avg": 0.20,
        "vote_count": 0.15,
        "visibility": 0.15,
        "engagement": 0.10,
        "reception": 0.15,
    }

    weighted = sum(pieces[col].fillna(pieces.mean(axis=1)) * w for col, w in weights.items())
    df["business_value_score"] = weighted.round(2)

    return df


def estimate_synopsis_business_value(similar_df):
    if similar_df.empty:
        return np.nan

    score_col = None
    if "predicted_business_value" in similar_df.columns and similar_df["predicted_business_value"].notna().any():
        score_col = "predicted_business_value"
    elif "business_value_score" in similar_df.columns and similar_df["business_value_score"].notna().any():
        score_col = "business_value_score"

    if score_col is None:
        return np.nan

    work = similar_df[[score_col, "similarity_score"]].copy()
    work = work.dropna()

    if work.empty:
        return np.nan

    weights = work["similarity_score"].clip(lower=0.001)
    value = np.average(work[score_col], weights=weights)
    return round(float(value), 2)


def business_value_band(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 75:
        return "High commercial potential"
    if score >= 55:
        return "Medium-high potential"
    if score >= 40:
        return "Moderate potential"
    return "Lower potential"


def dataset_health_text(df):
    parts = []
    parts.append(f"{len(df):,} titles")
    if "content_type" in df.columns:
        parts.append(f"{df['content_type'].nunique()} content types")
    if "source" in df.columns:
        parts.append(f"{df['source'].nunique()} sources")
    if "release_year" in df.columns and df["release_year"].notna().any():
        parts.append(
            f"years {int(df['release_year'].min())}–{int(df['release_year'].max())}"
        )
    return " · ".join(parts)


def choose_dataset_file():
    for candidate in DEFAULT_DATASET_CANDIDATES:
        if Path(candidate).exists():
            return candidate
    return None


# ──────────────────────────────────────────────────────────────────────────────
# DATA
# ──────────────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    dataset_path = choose_dataset_file()
    if dataset_path is None:
        raise FileNotFoundError(
            "No dataset found. Please place one of these files in the app folder: "
            "final_streaming_dataset.csv, streamlit_ready_dataset.csv, all_streaming_titles.csv"
        )

    df = pd.read_csv(dataset_path)

    # Core cleanup
    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    numeric_cols = [
        "popularity", "vote_average", "vote_count",
        "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value",
        "topic_diversity_score", "runtime_final", "cluster",
        "audience_reception_score", "freshness_score",
        "release_year"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    text_cols = [
        "title", "overview", "overview_clean", "genre_names",
        "content_type", "source", "original_language",
        "marketing_segment", "cluster_label", "production_companies",
        "network", "status", "web_channel", "top_topics"
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    if "genre_names" in df.columns:
        df["genre_list"] = df["genre_names"].apply(parse_genres)
    else:
        df["genre_list"] = [[] for _ in range(len(df))]

    if "overview_clean" not in df.columns:
        df["overview_clean"] = df["overview"].fillna("").apply(clean_text)
    else:
        df["overview_clean"] = df["overview_clean"].fillna("").apply(clean_text)

    if "title" not in df.columns:
        df["title"] = "Untitled"

    if "content_type" not in df.columns:
        df["content_type"] = "unknown"

    df["title_key"] = df["title"].fillna("").astype(str).str.lower().str.strip()
    df["detected_topics_auto"] = df["overview"].fillna("").apply(detect_topics_from_synopsis)
    df["detected_topics_text"] = df["detected_topics_auto"].apply(lambda x: ", ".join(x) if x else "")

    df = ensure_business_value_score(df)

    if "predicted_business_value" not in df.columns:
        df["predicted_business_value"] = np.nan

    if "cluster_label" not in df.columns:
        df["cluster_label"] = ""

    if "marketing_segment" not in df.columns:
        df["marketing_segment"] = ""

    return df, dataset_path


@st.cache_data
def prepare_similarity(df):
    work_df = df.copy()

    work_df["topics_text"] = work_df.apply(
        lambda row: " ".join(get_active_topics_from_row(row)) if get_active_topics_from_row(row) else row.get("detected_topics_text", ""),
        axis=1
    )

    genre_text = work_df["genre_list"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
    work_df["similarity_text"] = (
        work_df["title"].fillna("") + " "
        + genre_text.fillna("") + " "
        + work_df["overview_clean"].fillna("") + " "
        + work_df["topics_text"].fillna("") + " "
        + work_df["original_language"].fillna("")
    ).str.lower()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=6000,
        ngram_range=(1, 2)
    )
    matrix = vectorizer.fit_transform(work_df["similarity_text"])

    return work_df, matrix


def get_similar_titles(df_similarity, matrix, selected_title, top_n=10):
    matches = df_similarity.index[df_similarity["title"] == selected_title].tolist()
    if not matches:
        return pd.DataFrame()

    idx = matches[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()

    sim_df = df_similarity.copy()
    sim_df["similarity_score"] = sims
    sim_df = sim_df[sim_df.index != idx].copy()

    sort_cols = ["similarity_score", "business_value_score", "vote_average", "popularity"]
    sort_cols = [c for c in sort_cols if c in sim_df.columns]

    rank_cols = [
        "title", "content_type", "genre_names", "release_year",
        "vote_average", "popularity", "visibility_score",
        "engagement_score", "business_value_score",
        "predicted_business_value", "marketing_segment",
        "cluster_label", "detected_topics_text", "similarity_score", "overview"
    ]
    existing_cols = [c for c in rank_cols if c in sim_df.columns]

    return sim_df.sort_values(
        by=sort_cols,
        ascending=[False] * len(sort_cols)
    )[existing_cols].head(top_n)


def get_top_similar_titles_from_synopsis(user_synopsis, content_type, df_similarity, matrix, top_n=5):
    working = df_similarity.copy()

    if content_type and content_type.lower() != "all" and "content_type" in working.columns:
        working = working[
            working["content_type"].astype(str).str.lower() == content_type.lower()
        ].copy()

    if working.empty:
        return pd.DataFrame()

    user_text = clean_text(user_synopsis)
    user_topics = set(detect_topics_from_synopsis(user_synopsis))

    user_vec = matrix.__class__(matrix.shape[0])  # placeholder not used directly

    # Need to rebuild on filtered subset from existing vectorizer behavior:
    # simplest safe path: refit a local vectorizer on filtered rows
    local_vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=6000,
        ngram_range=(1, 2)
    )
    local_matrix = local_vectorizer.fit_transform(working["similarity_text"])
    user_vec = local_vectorizer.transform([user_text])
    text_sims = cosine_similarity(user_vec, local_matrix).flatten()

    working["text_similarity"] = text_sims

    def topic_overlap_score(item_topics_text):
        item_topics = set([x.strip() for x in str(item_topics_text).split(",") if x.strip()])
        if not user_topics or not item_topics:
            return 0
        return len(user_topics.intersection(item_topics)) / max(len(user_topics), 1)

    working["topic_similarity"] = working["detected_topics_text"].apply(topic_overlap_score)

    working["similarity_score"] = (
        0.80 * working["text_similarity"] +
        0.20 * working["topic_similarity"]
    )

    cols_to_show = [
        col for col in [
            "title",
            "content_type",
            "genre_names",
            "original_language",
            "release_year",
            "overview",
            "business_value_score",
            "predicted_business_value",
            "marketing_segment",
            "cluster_label",
            "detected_topics_text",
            "text_similarity",
            "topic_similarity",
            "similarity_score"
        ] if col in working.columns
    ]

    result = working.sort_values(
        by=["similarity_score", "business_value_score"],
        ascending=[False, False]
    ).head(top_n)

    return result[cols_to_show]


def get_rank(df_all, title, col):
    if col not in df_all.columns:
        return None, None

    rank_df = df_all[["title", col]].dropna().sort_values(by=col, ascending=False).reset_index(drop=True)
    rank_df.index = rank_df.index + 1
    matches = rank_df[rank_df["title"] == title]

    if matches.empty:
        return None, len(rank_df)

    return int(matches.index[0]), len(rank_df)


# ──────────────────────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────────────────────
df, dataset_path = load_data()
df_similarity, sim_matrix = prepare_similarity(df)


# ──────────────────────────────────────────────────────────────────────────────
# HEADER
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("## 🎬 Streaming Marketing Intelligence Dashboard")

st.markdown(f"""
<div class="objective-card">
    <div class="section-title">Marketing Objective</div>
    <div class="muted">
        Support content marketing and acquisition decisions by identifying which titles show stronger commercial potential,
        what themes are most attractive, and which existing titles can serve as relevant comparables for a new idea or synopsis.
    </div>
    <br>
    <div class="muted">
        <strong>Primary use cases:</strong> content benchmarking · similarity search · idea validation · business value estimation · marketing storytelling
    </div>
</div>
""", unsafe_allow_html=True)

st.caption(f"Dataset loaded: `{dataset_path}` · {dataset_health_text(df)}")
st.divider()


# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────────────────────────────────────
st.sidebar.header("Filters")

content_options_real = sorted([x for x in df["content_type"].dropna().astype(str).unique().tolist() if x != ""])
content_options = ["All"] + content_options_real
selected_content = st.sidebar.selectbox("Content type", options=content_options, index=0)

source_options_real = sorted([x for x in df["source"].dropna().astype(str).unique().tolist() if x != ""]) if "source" in df.columns else []
source_options = ["All"] + source_options_real
selected_source = st.sidebar.selectbox("Source", options=source_options, index=0)

language_options_real = sorted([x for x in df["original_language"].dropna().astype(str).unique().tolist() if x != ""]) if "original_language" in df.columns else []
selected_languages = st.sidebar.multiselect(
    "Languages",
    options=["All"] + language_options_real,
    default=["All"]
)

all_detected_topics = sorted({
    topic
    for topics in df["detected_topics_auto"]
    for topic in topics
})
selected_topics = st.sidebar.multiselect(
    "Detected themes",
    options=all_detected_topics,
    default=[]
)

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider(
        "Release year range",
        min_value=min_year,
        max_value=max_year,
        value=(min_year, max_year)
    )
else:
    year_range = None

st.sidebar.divider()
st.sidebar.markdown("### Quick title search")

sidebar_title_options = sorted(df["title"].dropna().astype(str).unique().tolist())
sidebar_selected_title = st.sidebar.selectbox(
    "Search a title",
    options=sidebar_title_options,
    index=None,
    placeholder="Start typing..."
)


# ──────────────────────────────────────────────────────────────────────────────
# FILTERS
# ──────────────────────────────────────────────────────────────────────────────
filtered_df = df.copy()

if selected_content != "All" and "content_type" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["content_type"] == selected_content]

if selected_source != "All" and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"] == selected_source]

if "All" not in selected_languages and selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]

if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")
    ]

if selected_topics:
    filtered_df = filtered_df[
        filtered_df["detected_topics_auto"].apply(
            lambda x: any(topic in x for topic in selected_topics)
        )
    ]


# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR QUICK RESULT
# ──────────────────────────────────────────────────────────────────────────────
if sidebar_selected_title:
    sidebar_row = df[df["title"] == sidebar_selected_title].head(1)
    if not sidebar_row.empty:
        row = sidebar_row.iloc[0]

        st.sidebar.markdown("#### Title snapshot")
        st.sidebar.markdown(
            f"**Type:** {row.get('content_type', 'N/A')}  \n"
            f"**Year:** {int(row['release_year']) if pd.notna(row.get('release_year')) else 'N/A'}  \n"
            f"**Language:** {row.get('original_language', 'N/A')}"
        )

        st.sidebar.markdown("#### Overview")
        overview_text = row.get("overview", "")
        if overview_text:
            st.sidebar.caption(overview_text[:350] + ("..." if len(overview_text) > 350 else ""))
        else:
            st.sidebar.caption("No overview available.")

        st.sidebar.markdown("#### Performance")
        st.sidebar.metric("Visibility", metric_str(row.get("visibility_score", np.nan)))
        st.sidebar.metric("Engagement", metric_str(row.get("engagement_score", np.nan)))
        st.sidebar.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))
        if pd.notna(row.get("predicted_business_value", np.nan)):
            st.sidebar.metric("Predicted BV", metric_str(row.get("predicted_business_value", np.nan)))

        active_topics = get_active_topics_from_row(row)
        if not active_topics:
            active_topics = row.get("detected_topics_auto", [])
        st.sidebar.markdown("#### Themes")
        st.sidebar.caption(", ".join(active_topics) if active_topics else "Not detected")


# ──────────────────────────────────────────────────────────────────────────────
# KPI STRIP
# ──────────────────────────────────────────────────────────────────────────────
st.markdown(f"### Filtered catalog: {filtered_df.shape[0]:,} titles")

total_titles = len(filtered_df)
avg_popularity = filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan
avg_vote = filtered_df["vote_average"].mean() if "vote_average" in filtered_df.columns else np.nan
avg_business = filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan
avg_visibility = filtered_df["visibility_score"].mean() if "visibility_score" in filtered_df.columns else np.nan

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total titles", f"{total_titles:,}")
k2.metric("Avg popularity", metric_str(avg_popularity))
k3.metric("Avg rating", metric_str(avg_vote))
k4.metric("Avg business value", metric_str(avg_business))

k5, k6, k7, k8 = st.columns(4)
k5.metric("Avg visibility", metric_str(avg_visibility))
k6.metric("Movies", f"{int((filtered_df['content_type'].astype(str).str.lower() == 'movie').sum()):,}" if "content_type" in filtered_df.columns else "N/A")
k7.metric("Series / TV", f"{int((filtered_df['content_type'].astype(str).str.lower().isin(['tv', 'series'])).sum()):,}" if "content_type" in filtered_df.columns else "N/A")
k8.metric("Sources", f"{filtered_df['source'].nunique():,}" if "source" in filtered_df.columns else "N/A")

st.divider()


# ──────────────────────────────────────────────────────────────────────────────
# TABS
# ──────────────────────────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
    "📊 Executive Overview",
    "🎯 Marketing Performance",
    "🧠 Audience & Themes",
    "🔎 Similar Titles Finder",
    "📝 Synopsis Opportunity Lab",
    "🗂 Dataset Explorer"
])


# ══════════════════════════════════════════════════════════════════════════════
# TAB 1 – EXECUTIVE OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════
with tab1:
    st.subheader("Executive overview")

    c1, c2 = st.columns(2)

    with c1:
        if "content_type" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["content_type"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_1)
            ax.set_title("Titles by content type", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Count", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "source" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["source"].value_counts().head(10)
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_2)
            ax.set_title("Top sources", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Count", color=TEXT)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "popularity" in filtered_df.columns and filtered_df["popularity"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["popularity"].dropna(), bins=30, color=ACCENT_3)
            ax.set_title("Popularity distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Popularity", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c4:
        if "business_value_score" in filtered_df.columns and filtered_df["business_value_score"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["business_value_score"].dropna(), bins=30, color=ACCENT_5)
            ax.set_title("Business value distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Business Value Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    if "release_year" in filtered_df.columns and filtered_df["release_year"].notna().any():
        yearly = filtered_df["release_year"].dropna().astype(int).value_counts().sort_index()
        fig, ax = make_dark_fig((12, 4))
        ax.plot(yearly.index, yearly.values)
        ax.set_title("Catalog trend by release year", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        ax.set_xlabel("Release year", color=TEXT)
        ax.set_ylabel("Titles", color=TEXT)
        st.pyplot(fig)
        plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# TAB 2 – MARKETING PERFORMANCE
# ══════════════════════════════════════════════════════════════════════════════
with tab2:
    st.subheader("Marketing performance analysis")

    metric_map = {
        "Visibility": "visibility_score",
        "Engagement": "engagement_score",
        "Business Value": "business_value_score",
        "Popularity": "popularity",
        "Audience Reception": "audience_reception_score"
    }

    available_metric_labels = [k for k, v in metric_map.items() if v in filtered_df.columns]
    selected_metric_label = st.radio(
        "Highlighted KPI",
        options=available_metric_labels,
        horizontal=True,
        index=available_metric_labels.index("Business Value") if "Business Value" in available_metric_labels else 0
    )
    highlight_metric = metric_map[selected_metric_label]

    c1, c2 = st.columns(2)

    with c1:
        top_df = filtered_df[["title", highlight_metric]].dropna().sort_values(
            by=highlight_metric, ascending=False
        ).head(10)

        if not top_df.empty:
            fig, ax = make_dark_fig((8, 5))
            ax.barh(top_df["title"][::-1], top_df[highlight_metric][::-1], color=ACCENT_1)
            ax.set_title(f"Top 10 by {selected_metric_label}", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "marketing_segment" in filtered_df.columns and filtered_df["marketing_segment"].astype(str).str.strip().ne("").any():
            seg = filtered_df["marketing_segment"].replace("", "Unknown").value_counts().head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.bar(seg.index, seg.values, color=ACCENT_2)
            ax.set_title("Marketing segment distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("No `marketing_segment` column available in the current dataset.")

    if all(c in filtered_df.columns for c in ["popularity", "business_value_score"]):
        sample_df = filtered_df[["popularity", "business_value_score"]].dropna().sample(
            min(5000, len(filtered_df[["popularity", "business_value_score"]].dropna())),
            random_state=42
        )
        fig, ax = make_dark_fig((10, 5))
        ax.scatter(sample_df["popularity"], sample_df["business_value_score"], alpha=0.35)
        ax.set_title("Popularity vs Business Value", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        ax.set_xlabel("Popularity", color=TEXT)
        ax.set_ylabel("Business Value Score", color=TEXT)
        st.pyplot(fig)
        plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# TAB 3 – AUDIENCE & THEMES
# ══════════════════════════════════════════════════════════════════════════════
with tab3:
    st.subheader("Audience and thematic signals")

    c1, c2 = st.columns(2)

    with c1:
        topic_counts = {}
        for topic_list in filtered_df["detected_topics_auto"]:
            for topic in topic_list:
                topic_counts[topic] = topic_counts.get(topic, 0) + 1

        if topic_counts:
            topic_series = pd.Series(topic_counts).sort_values(ascending=False).head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.barh(topic_series.index[::-1], topic_series.values[::-1], color=ACCENT_3)
            ax.set_title("Top detected themes", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Count", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "original_language" in filtered_df.columns:
            lang_counts = filtered_df["original_language"].replace("", "Unknown").value_counts().head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.bar(lang_counts.index, lang_counts.values, color=ACCENT_4)
            ax.set_title("Top languages", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    if "genre_names" in filtered_df.columns:
        exploded = (
            filtered_df.assign(genre_item=filtered_df["genre_list"])
            .explode("genre_item")
        )
        exploded = exploded[exploded["genre_item"].notna() & (exploded["genre_item"] != "")]
        if not exploded.empty:
            top_genres = exploded["genre_item"].value_counts().head(12)
            fig, ax = make_dark_fig((10, 5))
            ax.barh(top_genres.index[::-1], top_genres.values[::-1], color=ACCENT_5)
            ax.set_title("Top genres", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Count", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# TAB 4 – SIMILAR TITLES FINDER
# ══════════════════════════════════════════════════════════════════════════════
with tab4:
    st.subheader("Similar titles finder")

    available_titles = sorted(filtered_df["title"].dropna().astype(str).unique().tolist())

    if not available_titles:
        st.warning("No titles available under the current filters.")
    else:
        selected_title = st.selectbox(
            "Start typing and select a title",
            options=available_titles,
            index=None,
            placeholder="Type a movie or series title..."
        )

        similar_df = pd.DataFrame()

        if selected_title:
            selected_row = filtered_df[filtered_df["title"] == selected_title].head(1)

            if not selected_row.empty:
                row = selected_row.iloc[0]

                rank_pop, total_pop = get_rank(df, selected_title, "popularity")
                rank_vote, total_vote = get_rank(df, selected_title, "vote_average")
                rank_business, total_business = get_rank(df, selected_title, "business_value_score")

                st.markdown(f"#### {selected_title}")

                info1, info2, info3, info4 = st.columns(4)
                info1.metric("Type", row.get("content_type", "N/A"))
                info2.metric("Year", int(row["release_year"]) if pd.notna(row.get("release_year")) else "N/A")
                info3.metric("Rating", metric_str(row.get("vote_average", np.nan)))
                info4.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))

                st.markdown(
                    f"**Genres:** {row.get('genre_names', 'N/A')}  \n"
                    f"**Language:** {row.get('original_language', 'N/A')}  \n"
                    f"**Marketing Segment:** {row.get('marketing_segment', 'N/A')}  \n"
                    f"**Cluster:** {row.get('cluster_label', 'N/A') if str(row.get('cluster_label', '')).strip() else 'N/A'}"
                )

                active_topics = get_active_topics_from_row(row)
                if not active_topics:
                    active_topics = row.get("detected_topics_auto", [])
                st.markdown(f"**Detected themes:** {', '.join(active_topics) if active_topics else 'Not detected'}")

                if row.get("overview"):
                    st.markdown("**Overview**")
                    st.write(row["overview"])

                r1, r2, r3 = st.columns(3)
                r1.metric("Popularity rank", f"#{rank_pop}" if rank_pop else "N/A", delta=f"of {total_pop}" if total_pop else None)
                r2.metric("Rating rank", f"#{rank_vote}" if rank_vote else "N/A", delta=f"of {total_vote}" if total_vote else None)
                r3.metric("Business rank", f"#{rank_business}" if rank_business else "N/A", delta=f"of {total_business}" if total_business else None)

                st.divider()

                similar_df = get_similar_titles(df_similarity, sim_matrix, selected_title, top_n=10)

                st.markdown("#### Most similar titles")

                if similar_df.empty:
                    st.info("No similar titles found.")
                else:
                    fig, ax = make_dark_fig((9, 5))
                    plot_df = similar_df.head(8).copy()
                    ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_1)
                    ax.set_title("Top similarity", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                    ax.set_xlabel("Similarity Score", color=TEXT)
                    st.pyplot(fig)
                    plt.close(fig)

                    display_df = similar_df.copy()
                    if "similarity_score" in display_df.columns:
                        display_df["similarity_score"] = display_df["similarity_score"].round(3)

                    st.dataframe(display_df, use_container_width=True)

        if not similar_df.empty:
            csv_similar = similar_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button(
                label="Download similar titles",
                data=csv_similar,
                file_name="similar_titles_results.csv",
                mime="text/csv"
            )


# ══════════════════════════════════════════════════════════════════════════════
# TAB 5 – SYNOPSIS OPPORTUNITY LAB
# ══════════════════════════════════════════════════════════════════════════════
with tab5:
    st.subheader("Synopsis opportunity lab")
    st.markdown(
        """
        Paste a new synopsis and the app will:
        1. detect its likely themes,
        2. retrieve the most similar titles in the dataset,
        3. estimate a likely business value based on comparable content.
        """
    )

    user_type = st.selectbox(
        "Content type",
        options=["All", "movie", "tv"],
        index=0
    )

    user_synopsis = st.text_area(
        "Paste the synopsis",
        height=180,
        placeholder="Example: A young journalist uncovers a political corruption network while trying to protect her family and save her career..."
    )

    top_n = st.slider("Number of comparables", min_value=3, max_value=10, value=5)

    if st.button("Analyze synopsis"):
        if not user_synopsis or not user_synopsis.strip():
            st.warning("Please paste a synopsis first.")
        else:
            detected_topics = detect_topics_from_synopsis(user_synopsis)

            synopsis_results = get_top_similar_titles_from_synopsis(
                user_synopsis=user_synopsis,
                content_type=user_type,
                df_similarity=df_similarity,
                matrix=sim_matrix,
                top_n=top_n
            )

            estimated_bv = estimate_synopsis_business_value(synopsis_results)
            estimated_band = business_value_band(estimated_bv)

            st.session_state["synopsis_results"] = synopsis_results
            st.session_state["user_synopsis_text"] = user_synopsis
            st.session_state["user_synopsis_topics"] = detected_topics
            st.session_state["user_synopsis_type"] = user_type
            st.session_state["estimated_bv"] = estimated_bv
            st.session_state["estimated_band"] = estimated_band

    if "synopsis_results" in st.session_state:
        synopsis_results = st.session_state["synopsis_results"]
        detected_topics = st.session_state.get("user_synopsis_topics", [])
        user_synopsis_saved = st.session_state.get("user_synopsis_text", "")
        user_type_saved = st.session_state.get("user_synopsis_type", "All")
        estimated_bv = st.session_state.get("estimated_bv", np.nan)
        estimated_band = st.session_state.get("estimated_band", "Unknown")

        a1, a2, a3 = st.columns(3)
        a1.metric("Detected themes", f"{len(detected_topics)}")
        a2.metric("Estimated Business Value", metric_str(estimated_bv))
        a3.metric("Commercial assessment", estimated_band)

        st.markdown("### Detected themes")
        st.write(", ".join(detected_topics) if detected_topics else "No clear themes detected.")

        st.markdown("### Similar titles")
        if synopsis_results.empty:
            st.info("No comparable titles were found for the current filters.")
        else:
            fig, ax = make_dark_fig((9, 5))
            plot_df = synopsis_results.head(8).copy()
            ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_2)
            ax.set_title("Top comparable titles by similarity", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Similarity Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

            for i, (_, row) in enumerate(synopsis_results.iterrows(), start=1):
                st.markdown(f"#### #{i} - {row.get('title', 'Unknown title')}")
                st.write(f"**Type:** {row.get('content_type', 'N/A')}")
                st.write(f"**Genres:** {row.get('genre_names', 'N/A')}")
                st.write(f"**Language:** {row.get('original_language', 'N/A')}")
                st.write(f"**Year:** {row.get('release_year', 'N/A')}")
                st.write(f"**Text similarity:** {row.get('text_similarity', 0):.3f}")
                st.write(f"**Topic similarity:** {row.get('topic_similarity', 0):.3f}")
                st.write(f"**Final similarity score:** {row.get('similarity_score', 0):.3f}")

                if "business_value_score" in row and pd.notna(row["business_value_score"]):
                    st.write(f"**Business Value Score:** {row['business_value_score']:.2f}")

                if "predicted_business_value" in row and pd.notna(row["predicted_business_value"]):
                    st.write(f"**Predicted Business Value:** {row['predicted_business_value']:.2f}")

                if "marketing_segment" in row and str(row.get("marketing_segment", "")).strip():
                    st.write(f"**Marketing Segment:** {row.get('marketing_segment')}")

                st.write(f"**Overview:** {row.get('overview', 'N/A')}")
                st.markdown("---")

            st.markdown("### Opportunity readout")
            if pd.notna(estimated_bv):
                if estimated_bv >= 75:
                    st.success(
                        "This synopsis resembles titles with strong commercial potential. "
                        "It may be a good candidate for premium positioning, stronger promotion, or greenlight discussion."
                    )
                elif estimated_bv >= 55:
                    st.info(
                        "This synopsis resembles titles with medium-high potential. "
                        "It could perform well depending on packaging, cast, timing, and campaign strategy."
                    )
                elif estimated_bv >= 40:
                    st.warning(
                        "This synopsis resembles titles with moderate potential. "
                        "It may require sharper positioning or a more differentiated marketing angle."
                    )
                else:
                    st.error(
                        "This synopsis resembles titles with lower estimated commercial value in the current dataset."
                    )
            else:
                st.info("Business value estimation could not be calculated from the available comparable titles.")

            export_df = synopsis_results.copy()
            export_df.insert(0, "input_synopsis", user_synopsis_saved)
            export_df.insert(1, "input_content_type", user_type_saved)
            export_df.insert(2, "detected_topics", ", ".join(detected_topics))
            export_df.insert(3, "estimated_business_value", estimated_bv)
            export_df.insert(4, "estimated_business_value_band", estimated_band)

            csv_synopsis = export_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button(
                label="Download synopsis analysis",
                data=csv_synopsis,
                file_name="synopsis_opportunity_results.csv",
                mime="text/csv"
            )


# ══════════════════════════════════════════════════════════════════════════════
# TAB 6 – DATASET EXPLORER
# ══════════════════════════════════════════════════════════════════════════════
with tab6:
    st.subheader("Dataset explorer")

    explorer_cols = [c for c in [
        "title", "content_type", "genre_names", "release_year",
        "original_language", "source", "popularity", "vote_average",
        "vote_count", "visibility_score", "engagement_score",
        "audience_reception_score", "business_value_score", "predicted_business_value",
        "marketing_segment", "cluster_label", "detected_topics_text", "overview"
    ] if c in filtered_df.columns]

    st.dataframe(filtered_df[explorer_cols].head(200), use_container_width=True)

    csv = filtered_df.to_csv(index=False).encode("utf-8-sig")
    st.download_button(
        label="Download filtered dataset",
        data=csv,
        file_name="filtered_streaming_dataset.csv",
        mime="text/csv"
    )'''


app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    webbrowser.open(url)
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()

Starting Streamlit on http://127.0.0.1:56211 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:56211

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:56211


2026-08-07 01:47:56.621 Pandas DataFrame hash failed. Falling back to pickling the object.
Traceback (most recent call last):
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/streamlit/runtime/caching/hashing.py", line 454, in _to_bytes
    values_hash_bytes = self.to_bytes(hash_pandas_object(df_obj))
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 174, in hash_pandas_object
    h = combine_hash_arrays(hashes, num_items)
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 72, in combine_hash_arrays
    for i, a in enumerate(arrays):
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 173, in <genexpr>
    hashes = (x for x in _hashes)
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 154, in <genexpr>
    hash_array(series._values, encoding, hash_key, categorize)
  File 

# not fav

In [ ]:
import socket
import subprocess
import time
import urllib.request
import sys
import webbrowser
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False

streamlit_code = r'''
import ast
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# =============================================================================
# CONFIG
# =============================================================================
st.set_page_config(
    page_title="Streaming Marketing Intelligence",
    page_icon="🎬",
    layout="wide",
)

CARD_BG = "#1E293B"
FIG_BG = "#0F172A"
TEXT = "white"
ACCENT_1 = "#6366F1"
ACCENT_2 = "#10B981"
ACCENT_3 = "#F59E0B"
ACCENT_4 = "#EF4444"
ACCENT_5 = "#22C55E"
MUTED = "#94A3B8"

DATASET_CANDIDATES = [
    "DATA/PROCESSED/all_streaming_titles.csv",
]

MODEL_DIR_CANDIDATES = [
    "MODELS",
    "./MODELS",
    "/mnt/data/MODELS",
]

RF_CANDIDATES = ["rf_model.pkl", "rf_model.joblib"]
KMEANS_EV_CANDIDATES = ["kmeans_engage_vis.joblib"]
KMEANS_FINAL_CANDIDATES = ["kmeans_final.joblib"]
SCALER_EV_CANDIDATES = ["scaler_ev.joblib"]
KMEANS_SCALER_CANDIDATES = ["kmeans_scaler.joblib"]

# Inferred from your feature importance plot.
# If your trained model expects different columns, edit this list.
RF_FEATURES = [
    "vote_average",
    "audience_reception_score",
    "release_year",
    "vote_count",
    "popularity",
    "visibility_score",
    "engagement_score",
    "topic_diversity_score",
    "runtime_final",
]

TOPIC_KEYWORDS = {
    "lgbtq": [
        "gay", "lesbian", "lgbt", "trans", "queer", "bisexual",
        "nonbinary", "coming out", "drag", "identity"
    ],

    "politics": [
        "politics", "president", "government", "election", "policy",
        "senate", "congress", "campaign", "minister", "dictator",
        "democracy", "corruption", "power", "state"
    ],

    "climate_environment": [
        "climate", "environment", "global warming", "pollution",
        "ecology", "sustainability", "nature", "forest", "wildlife",
        "environmental disaster", "carbon", "climate crisis"
    ],

    "war_military": [
        "war", "battle", "army", "soldier", "military", "conflict",
        "weapon", "navy", "air force", "commander", "resistance",
        "invasion", "combat", "veteran"
    ],

    "family": [
        "family", "mother", "father", "parent", "children", "child",
        "home", "siblings", "brother", "sister", "marriage",
        "divorce", "parenthood", "relatives"
    ],

    "crime": [
        "crime", "murder", "police", "detective", "investigation",
        "killer", "gang", "mafia", "cartel", "robbery", "heist",
        "forensics", "prison", "criminal", "underworld"
    ],

    "policial": [
        "police", "cop", "officer", "detective", "agent", "rookie",
        "investigation", "investigator", "inspector", "sergeant",
        "lieutenant", "captain", "chief", "precinct", "partner",
        "case", "crime scene", "forensics", "suspect", "witness",
        "interrogation", "undercover", "surveillance", "homicide",
        "manhunt", "law enforcement", "special unit", "task force",
        "fbi", "cia", "narcotics", "patrol", "unit"
    ],

    "romance": [
        "love", "romance", "relationship", "couple", "passion",
        "heartbreak", "dating", "affair", "wedding", "breakup",
        "soulmate", "jealousy"
    ],

    "technology": [
        "technology", "ai", "artificial intelligence", "robot",
        "future", "cyber", "computer", "hacker", "internet",
        "virtual reality", "machine", "automation", "surveillance"
    ],

    "mental_health": [
        "depression", "anxiety", "trauma", "therapy", "mental",
        "psychological", "stress", "grief", "addiction", "bipolar",
        "schizophrenia", "panic", "healing", "psychiatric"
    ],

    "coming_of_age": [
        "teen", "adolescent", "growing up", "school", "youth",
        "friendship", "identity", "first love", "high school",
        "college", "self-discovery", "maturity"
    ],

    "social_issues": [
        "racism", "inequality", "poverty", "discrimination", "justice",
        "migration", "sexism", "class", "homophobia", "xenophobia",
        "oppression", "human rights", "activism"
    ],

    "fantasy_supernatural": [
        "magic", "witch", "dragon", "supernatural", "monster",
        "curse", "fantasy", "sorcery", "wizard", "demon", "ghost",
        "prophecy", "kingdom", "spell"
    ],

    "science_fiction": [
        "space", "alien", "spaceship", "future", "planet",
        "time travel", "parallel universe", "android", "mutation",
        "dystopia", "utopia", "interstellar", "extraterrestrial"
    ],

    "horror": [
        "horror", "fear", "haunted", "ghost", "possession",
        "slasher", "evil", "demon", "nightmare", "blood",
        "terror", "zombie", "paranormal"
    ],

    "thriller": [
        "thriller", "suspense", "mystery", "conspiracy", "chase",
        "secret", "obsession", "danger", "kidnapping", "betrayal",
        "tension", "survival"
    ],

    "action_adventure": [
        "action", "adventure", "hero", "mission", "explosion",
        "fight", "chase", "survival", "escape", "journey",
        "quest", "mercenary"
    ],

    "historical": [
        "history", "historical", "period drama", "king", "queen",
        "empire", "revolution", "civilization", "medieval",
        "ancient", "biographical", "royalty"
    ],

    "biography": [
        "biography", "biopic", "true story", "real life", "famous",
        "artist", "scientist", "politician", "athlete", "inventor"
    ],

    "sports": [
        "sport", "football", "soccer", "basketball", "baseball",
        "tennis", "boxing", "fighter", "competition", "tournament",
        "coach", "team", "championship"
    ],

    "music_performance": [
        "music", "band", "singer", "concert", "performance",
        "musician", "song", "dance", "ballet", "opera",
        "stage", "show business"
    ],

    "comedy": [
        "comedy", "funny", "humor", "satire", "parody",
        "awkward", "absurd", "joke", "laugh", "misunderstanding"
    ],

    "drama": [
        "drama", "emotional", "conflict", "sacrifice", "loss",
        "betrayal", "redemption", "personal struggle", "intense"
    ],

    "mystery": [
        "mystery", "secret", "clue", "disappearance", "puzzle",
        "unknown", "hidden truth", "unsolved", "investigation"
    ],

    "survival_disaster": [
        "survival", "disaster", "earthquake", "tsunami", "fire",
        "shipwreck", "plane crash", "apocalypse", "epidemic",
        "outbreak", "catastrophe"
    ],

    "religion_spirituality": [
        "religion", "faith", "god", "church", "priest", "spiritual",
        "belief", "miracle", "sacred", "ritual", "afterlife"
    ],

    "animation_family": [
        "animation", "animated", "family-friendly", "kids",
        "talking animals", "fairy tale", "adventure for children"
    ],

    "friendship": [
        "friendship", "best friends", "companionship", "bond",
        "loyalty", "group of friends", "reunion"
    ],

    "revenge": [
        "revenge", "vengeance", "payback", "betrayal", "justice",
        "retaliation", "avenger"
    ],

    "road_trip_journey": [
        "road trip", "journey", "travel", "on the road", "escape",
        "self-discovery", "destination"
    ],

    "school_university": [
        "school", "teacher", "student", "classroom", "university",
        "college", "campus", "exam", "bullying", "graduation"
    ],

    "work_business": [
        "work", "office", "career", "boss", "company", "business",
        "corporate", "startup", "ambition", "promotion", "colleague"
    ]
}


# =============================================================================
# STYLE
# =============================================================================
st.markdown(
    """
<style>
.block-container {
    padding-top: 1.2rem;
    padding-bottom: 2rem;
}
div[data-testid="stMetric"] {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    padding: 14px 16px;
    border-radius: 16px;
}
div[data-testid="stDataFrame"] {
    border-radius: 12px;
}
.objective-card {
    background: linear-gradient(135deg, #0F172A 0%, #111827 100%);
    border: 1px solid #1E293B;
    border-radius: 18px;
    padding: 18px 20px;
    margin-bottom: 16px;
}
.section-title {
    font-size: 1.1rem;
    font-weight: 700;
    color: white;
    margin-bottom: 0.5rem;
}
.muted {
    color: #94A3B8;
}
</style>
""",
    unsafe_allow_html=True,
)


# =============================================================================
# HELPERS
# =============================================================================
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9áéíóúñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def metric_str(value, decimals=2):
    return f"{value:.{decimals}f}" if pd.notna(value) else "N/A"


def parse_genres(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [g.strip() for g in text.split(",") if g.strip()]


def detect_topics_from_synopsis(text: str):
    text_clean = clean_text(text)
    found_topics = []
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword.lower() in text_clean for keyword in keywords):
            found_topics.append(topic)
    if not found_topics:
        found_topics.append("General / No clear topic")
    return found_topics


def get_active_topics_from_row(row):
    topics = []

    if "top_topics" in row.index and pd.notna(row.get("top_topics")):
        raw = str(row.get("top_topics")).strip()
        if raw:
            if "|" in raw:
                topics.extend([x.strip() for x in raw.split("|") if x.strip()])
            elif "," in raw:
                topics.extend([x.strip() for x in raw.split(",") if x.strip()])
            else:
                topics.append(raw)

    topic_like_cols = [c for c in row.index if c.startswith("topic_") and c != "topic_diversity_score"]
    for col in topic_like_cols:
        try:
            if float(row.get(col, 0)) == 1:
                topics.append(col.replace("topic_", "").replace("_", " ").title())
        except Exception:
            continue

    return sorted(set([t for t in topics if t]))


def make_dark_fig(figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(FIG_BG)
    ax.set_facecolor(CARD_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT)
    return fig, ax


def normalize_0_100(series: pd.Series) -> pd.Series:
    series = pd.to_numeric(series, errors="coerce")
    valid = series.dropna()
    if valid.empty:
        return pd.Series(np.nan, index=series.index)
    min_v = valid.min()
    max_v = valid.max()
    if min_v == max_v:
        return pd.Series(50.0, index=series.index)
    return ((series - min_v) / (max_v - min_v)) * 100


def business_value_band(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 75:
        return "High commercial potential"
    if score >= 55:
        return "Medium-high potential"
    if score >= 40:
        return "Moderate potential"
    return "Lower potential"


def choose_existing_path(candidates, base_dir=None):
    if base_dir is not None:
        for candidate in candidates:
            p = Path(base_dir) / candidate
            if p.exists():
                return p
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    return None


def choose_dataset_file():
    return choose_existing_path(DATASET_CANDIDATES)


def choose_model_dir():
    for candidate in MODEL_DIR_CANDIDATES:
        p = Path(candidate)
        if p.exists() and p.is_dir():
            return p
    return None


def dataset_health_text(df):
    parts = [f"{len(df):,} titles"]
    if "content_type" in df.columns:
        parts.append(f"{df['content_type'].nunique()} content types")
    if "source" in df.columns:
        parts.append(f"{df['source'].nunique()} sources")
    if "release_year" in df.columns and df["release_year"].notna().any():
        parts.append(f"years {int(df['release_year'].min())}–{int(df['release_year'].max())}")
    return " · ".join(parts)


def ensure_business_value_score(df: pd.DataFrame) -> pd.DataFrame:
    if "business_value_score" in df.columns and df["business_value_score"].notna().any():
        return df

    pop_norm = normalize_0_100(df["popularity"]) if "popularity" in df.columns else pd.Series(np.nan, index=df.index)
    vote_avg_norm = normalize_0_100(df["vote_average"]) if "vote_average" in df.columns else pd.Series(np.nan, index=df.index)
    vote_count_norm = normalize_0_100(df["vote_count"]) if "vote_count" in df.columns else pd.Series(np.nan, index=df.index)
    visibility = normalize_0_100(df["visibility_score"]) if "visibility_score" in df.columns else pd.Series(np.nan, index=df.index)
    engagement = normalize_0_100(df["engagement_score"]) if "engagement_score" in df.columns else pd.Series(np.nan, index=df.index)
    reception = normalize_0_100(df["audience_reception_score"]) if "audience_reception_score" in df.columns else pd.Series(np.nan, index=df.index)

    pieces = pd.concat([pop_norm, vote_avg_norm, vote_count_norm, visibility, engagement, reception], axis=1)
    pieces.columns = ["pop", "vote_avg", "vote_count", "visibility", "engagement", "reception"]
    row_mean = pieces.mean(axis=1)

    weights = {
        "pop": 0.25,
        "vote_avg": 0.20,
        "vote_count": 0.15,
        "visibility": 0.15,
        "engagement": 0.10,
        "reception": 0.15,
    }
    weighted = sum(pieces[col].fillna(row_mean) * w for col, w in weights.items())
    df["business_value_score"] = weighted.round(2)
    return df


def compute_visibility_score(df: pd.DataFrame) -> pd.DataFrame:
    if "visibility_score" in df.columns and df["visibility_score"].notna().any():
        return df
    components = []
    if "popularity" in df.columns:
        components.append(normalize_0_100(df["popularity"]))
    if "vote_count" in df.columns:
        components.append(normalize_0_100(df["vote_count"]))
    if components:
        df["visibility_score"] = pd.concat(components, axis=1).mean(axis=1).round(2)
    else:
        df["visibility_score"] = np.nan
    return df


def compute_engagement_score(df: pd.DataFrame) -> pd.DataFrame:
    if "engagement_score" in df.columns and df["engagement_score"].notna().any():
        return df
    components = []
    if "vote_average" in df.columns:
        components.append(normalize_0_100(df["vote_average"]))
    if "vote_count" in df.columns:
        components.append(normalize_0_100(df["vote_count"]))
    if components:
        df["engagement_score"] = pd.concat(components, axis=1).mean(axis=1).round(2)
    else:
        df["engagement_score"] = np.nan
    return df


def ensure_audience_reception(df: pd.DataFrame) -> pd.DataFrame:
    if "audience_reception_score" in df.columns and df["audience_reception_score"].notna().any():
        return df
    if "vote_average" in df.columns:
        df["audience_reception_score"] = normalize_0_100(df["vote_average"]).round(2)
    else:
        df["audience_reception_score"] = np.nan
    return df


def ensure_runtime_final(df: pd.DataFrame) -> pd.DataFrame:
    if "runtime_final" in df.columns and df["runtime_final"].notna().any():
        return df
    for col in ["runtime", "episode_run_time"]:
        if col in df.columns:
            df["runtime_final"] = pd.to_numeric(df[col], errors="coerce")
            return df
    df["runtime_final"] = np.nan
    return df


def ensure_topic_diversity(df: pd.DataFrame) -> pd.DataFrame:
    if "topic_diversity_score" in df.columns and df["topic_diversity_score"].notna().any():
        return df
    topic_like_cols = [c for c in df.columns if c.startswith("topic_") and c != "topic_diversity_score"]
    if topic_like_cols:
        df["topic_diversity_score"] = df[topic_like_cols].fillna(0).sum(axis=1)
    else:
        df["topic_diversity_score"] = df["detected_topics_auto"].apply(lambda x: len(x) if isinstance(x, list) else 0)
    return df


def safe_fill_model_inputs(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    X = df.copy()
    for col in feature_cols:
        if col not in X.columns:
            X[col] = np.nan
        X[col] = pd.to_numeric(X[col], errors="coerce")
        if X[col].notna().any():
            X[col] = X[col].fillna(X[col].median())
        else:
            X[col] = 0.0
    return X[feature_cols]


def feature_importance_frame(model, feature_names: list[str]) -> pd.DataFrame:
    if not hasattr(model, "feature_importances_"):
        return pd.DataFrame(columns=["feature", "importance"])
    return (
        pd.DataFrame({"feature": feature_names, "importance": model.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


def estimate_synopsis_business_value(similar_df: pd.DataFrame):
    if similar_df.empty:
        return np.nan

    score_col = None
    if "predicted_business_value" in similar_df.columns and similar_df["predicted_business_value"].notna().any():
        score_col = "predicted_business_value"
    elif "business_value_score" in similar_df.columns and similar_df["business_value_score"].notna().any():
        score_col = "business_value_score"

    if score_col is None:
        return np.nan

    work = similar_df[[score_col, "similarity_score"]].dropna().copy()
    if work.empty:
        return np.nan

    weights = work["similarity_score"].clip(lower=0.001)
    value = np.average(work[score_col], weights=weights)
    return round(float(value), 2)


def get_rank(df_all, title, col):
    if col not in df_all.columns:
        return None, None
    rank_df = df_all[["title", col]].dropna().sort_values(by=col, ascending=False).reset_index(drop=True)
    rank_df.index = rank_df.index + 1
    matches = rank_df[rank_df["title"] == title]
    if matches.empty:
        return None, len(rank_df)
    return int(matches.index[0]), len(rank_df)


# =============================================================================
# LOAD DATA
# =============================================================================
@st.cache_data
def load_data():
    dataset_path = choose_dataset_file()
    if dataset_path is None:
        raise FileNotFoundError(
            "No dataset found. Add one of these: streamlit_ready_dataset.csv, final_streaming_dataset.csv, all_streaming_titles.csv"
        )

    df = pd.read_csv(dataset_path)

    numeric_cols = [
        "popularity", "vote_average", "vote_count", "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value", "topic_diversity_score", "runtime_final",
        "cluster", "audience_reception_score", "release_year", "runtime",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    text_cols = [
        "title", "overview", "overview_clean", "genre_names", "content_type", "source",
        "original_language", "marketing_segment", "cluster_label", "production_companies", "top_topics"
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    if "title" not in df.columns:
        df["title"] = "Untitled"
    if "overview" not in df.columns:
        df["overview"] = ""
    if "content_type" not in df.columns:
        df["content_type"] = "unknown"

    df["overview_clean"] = df["overview_clean"].fillna("") if "overview_clean" in df.columns else df["overview"].fillna("")
    df["overview_clean"] = df["overview_clean"].apply(clean_text)
    df["genre_list"] = df["genre_names"].apply(parse_genres) if "genre_names" in df.columns else [[] for _ in range(len(df))]
    df["title_key"] = df["title"].astype(str).str.lower().str.strip()
    df["detected_topics_auto"] = df["overview"].fillna("").apply(detect_topics_from_synopsis)
    df["detected_topics_text"] = df["detected_topics_auto"].apply(lambda x: ", ".join(x) if x else "")

    df = compute_visibility_score(df)
    df = compute_engagement_score(df)
    df = ensure_audience_reception(df)
    df = ensure_runtime_final(df)
    df = ensure_topic_diversity(df)
    df = ensure_business_value_score(df)

    if "predicted_business_value" not in df.columns:
        df["predicted_business_value"] = np.nan
    if "marketing_segment" not in df.columns:
        df["marketing_segment"] = ""
    if "cluster_label" not in df.columns:
        df["cluster_label"] = ""

    return df, str(dataset_path)


@st.cache_resource
def load_models():
    model_dir = choose_model_dir()
    assets = {
        "model_dir": str(model_dir) if model_dir else None,
        "rf_model": None,
        "kmeans_ev": None,
        "kmeans_final": None,
        "scaler_ev": None,
        "kmeans_scaler": None,
        "rf_features": RF_FEATURES,
        "messages": [],
    }

    if model_dir is None:
        assets["messages"].append("MODELS folder not found. The app will run with analytics + similarity only.")
        return assets

    try:
        rf_path = choose_existing_path(RF_CANDIDATES, model_dir)
        if rf_path:
            assets["rf_model"] = joblib.load(rf_path)
            assets["messages"].append(f"Loaded Random Forest model: {rf_path.name}")
            if hasattr(assets["rf_model"], "feature_names_in_"):
                assets["rf_features"] = list(assets["rf_model"].feature_names_in_)
        else:
            assets["messages"].append("Random Forest model not found.")
    except Exception as exc:
        assets["messages"].append(f"Could not load Random Forest model: {exc}")

    for key, candidates in [
        ("kmeans_ev", KMEANS_EV_CANDIDATES),
        ("kmeans_final", KMEANS_FINAL_CANDIDATES),
        ("scaler_ev", SCALER_EV_CANDIDATES),
        ("kmeans_scaler", KMEANS_SCALER_CANDIDATES),
    ]:
        try:
            p = choose_existing_path(candidates, model_dir)
            if p:
                assets[key] = joblib.load(p)
                assets["messages"].append(f"Loaded {p.name}")
        except Exception as exc:
            assets["messages"].append(f"Could not load {key}: {exc}")

    return assets


@st.cache_data
def prepare_similarity(df: pd.DataFrame):
    work_df = df.copy()
    work_df["topics_text"] = work_df.apply(
        lambda row: " ".join(get_active_topics_from_row(row)) if get_active_topics_from_row(row) else row.get("detected_topics_text", ""),
        axis=1,
    )
    genre_text = work_df["genre_list"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
    work_df["similarity_text"] = (
        work_df["title"].fillna("") + " "
        + genre_text.fillna("") + " "
        + work_df["overview_clean"].fillna("") + " "
        + work_df["topics_text"].fillna("") + " "
        + work_df["original_language"].fillna("")
    ).str.lower()

    vectorizer = TfidfVectorizer(stop_words="english", max_features=6000, ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(work_df["similarity_text"])
    return work_df, vectorizer, matrix


def apply_cluster_models(df: pd.DataFrame, assets: dict) -> pd.DataFrame:
    out = df.copy()

    # KMeans for visibility/engagement style segmentation
    if assets.get("kmeans_ev") is not None:
        ev_features = [c for c in ["visibility_score", "engagement_score"] if c in out.columns]
        if len(ev_features) == 2:
            ev_input = out[ev_features].copy()
            ev_input = ev_input.fillna(ev_input.median(numeric_only=True)).fillna(0)
            try:
                if assets.get("scaler_ev") is not None:
                    ev_scaled = assets["scaler_ev"].transform(ev_input)
                else:
                    ev_scaled = ev_input.values
                out["cluster_ev"] = assets["kmeans_ev"].predict(ev_scaled)
            except Exception:
                pass

    # KMeans final segmentation
    if assets.get("kmeans_final") is not None:
        final_features = [c for c in [
            "visibility_score", "engagement_score", "business_value_score", "audience_reception_score"
        ] if c in out.columns]
        if len(final_features) >= 2:
            final_input = out[final_features].copy()
            final_input = final_input.fillna(final_input.median(numeric_only=True)).fillna(0)
            try:
                if assets.get("kmeans_scaler") is not None:
                    final_scaled = assets["kmeans_scaler"].transform(final_input)
                else:
                    final_scaled = final_input.values
                out["cluster_final"] = assets["kmeans_final"].predict(final_scaled)
            except Exception:
                pass

    return out


def apply_rf_predictions(df: pd.DataFrame, assets: dict) -> pd.DataFrame:
    out = df.copy()
    rf_model = assets.get("rf_model")
    if rf_model is None:
        return out

    feature_cols = assets.get("rf_features", RF_FEATURES)
    X = safe_fill_model_inputs(out, feature_cols)

    try:
        out["predicted_business_value"] = rf_model.predict(X)
    except Exception:
        # fallback to inferred columns if loading feature metadata failed
        fallback_X = safe_fill_model_inputs(out, RF_FEATURES)
        out["predicted_business_value"] = rf_model.predict(fallback_X)
    return out


def get_similar_titles(df_similarity, matrix, selected_title, top_n=10):
    matches = df_similarity.index[df_similarity["title"] == selected_title].tolist()
    if not matches:
        return pd.DataFrame()

    idx = matches[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()

    sim_df = df_similarity.copy()
    sim_df["similarity_score"] = sims
    sim_df = sim_df[sim_df.index != idx].copy()

    sort_cols = [c for c in ["similarity_score", "predicted_business_value", "business_value_score", "vote_average", "popularity"] if c in sim_df.columns]
    cols = [
        "title", "content_type", "genre_names", "release_year", "vote_average", "popularity",
        "visibility_score", "engagement_score", "business_value_score", "predicted_business_value",
        "marketing_segment", "cluster_label", "cluster_ev", "cluster_final", "detected_topics_text",
        "similarity_score", "overview",
    ]
    cols = [c for c in cols if c in sim_df.columns]

    return sim_df.sort_values(by=sort_cols, ascending=[False] * len(sort_cols))[cols].head(top_n)


def get_top_similar_titles_from_synopsis(user_synopsis, content_type, df_similarity, top_n=5):
    working = df_similarity.copy()
    if content_type and content_type.lower() != "all" and "content_type" in working.columns:
        working = working[working["content_type"].astype(str).str.lower() == content_type.lower()].copy()
    if working.empty:
        return pd.DataFrame()

    local_vectorizer = TfidfVectorizer(stop_words="english", max_features=6000, ngram_range=(1, 2))
    local_matrix = local_vectorizer.fit_transform(working["similarity_text"])
    user_vec = local_vectorizer.transform([clean_text(user_synopsis)])
    text_sims = cosine_similarity(user_vec, local_matrix).flatten()
    working["text_similarity"] = text_sims

    user_topics = set(detect_topics_from_synopsis(user_synopsis))

    def topic_overlap_score(item_topics_text):
        item_topics = set([x.strip() for x in str(item_topics_text).split(",") if x.strip()])
        if not user_topics or not item_topics:
            return 0
        return len(user_topics.intersection(item_topics)) / max(len(user_topics), 1)

    working["topic_similarity"] = working["detected_topics_text"].apply(topic_overlap_score)
    working["similarity_score"] = 0.80 * working["text_similarity"] + 0.20 * working["topic_similarity"]

    cols = [
        "title", "content_type", "genre_names", "original_language", "release_year", "overview",
        "business_value_score", "predicted_business_value", "marketing_segment", "cluster_label",
        "cluster_ev", "cluster_final", "detected_topics_text", "text_similarity", "topic_similarity",
        "similarity_score",
    ]
    cols = [c for c in cols if c in working.columns]

    return working.sort_values(by=["similarity_score", "predicted_business_value", "business_value_score"], ascending=[False, False, False])[cols].head(top_n)


def build_manual_rf_input(models_assets):
    features = models_assets.get("rf_features", RF_FEATURES)
    st.markdown("### Manual prediction input")
    st.caption("Use this for an existing or hypothetical title when you know the model features.")

    values = {}
    c1, c2, c3 = st.columns(3)
    cols = [c1, c2, c3]

    defaults = {
        "vote_average": 7.0,
        "audience_reception_score": 70.0,
        "release_year": 2022.0,
        "vote_count": 1000.0,
        "popularity": 25.0,
        "visibility_score": 50.0,
        "engagement_score": 50.0,
        "topic_diversity_score": 2.0,
        "runtime_final": 100.0,
    }

    for i, feature in enumerate(features):
        with cols[i % 3]:
            values[feature] = st.number_input(
                feature,
                value=float(defaults.get(feature, 0.0)),
                step=1.0,
            )

    if st.button("Predict with trained Random Forest"):
        rf_model = models_assets.get("rf_model")
        if rf_model is None:
            st.error("Random Forest model is not loaded.")
        else:
            X_new = pd.DataFrame([{col: values.get(col, 0.0) for col in features}])
            pred = rf_model.predict(X_new)[0]
            st.success(f"Predicted Business Value: {pred:.2f}")
            st.info(f"Commercial assessment: {business_value_band(pred)}")


# =============================================================================
# LOAD EVERYTHING
# =============================================================================
df, dataset_path = load_data()
models_assets = load_models()
df = apply_cluster_models(df, models_assets)
df = apply_rf_predictions(df, models_assets)
df_similarity, similarity_vectorizer, sim_matrix = prepare_similarity(df)


# =============================================================================
# HEADER
# =============================================================================
st.markdown("## 🎬 Streaming Marketing Intelligence Dashboard")
st.markdown(
    """
<div class="objective-card">
    <div class="section-title">Marketing Objective</div>
    <div class="muted">
        Support content marketing and acquisition decisions by combining descriptive analytics,
        clustering, title similarity, and trained machine learning models to estimate commercial potential.
    </div>
    <br>
    <div class="muted">
        <strong>Includes:</strong> EDA · theme detection · similarity search · KMeans segmentation · Random Forest business value prediction
    </div>
</div>
""",
    unsafe_allow_html=True,
)
st.caption(f"Dataset loaded: `{dataset_path}` · {dataset_health_text(df)}")

with st.expander("Model loading status", expanded=False):
    st.write(f"Model directory: {models_assets.get('model_dir')}")
    for msg in models_assets.get("messages", []):
        st.write(f"- {msg}")
    st.write(f"RF features in use: {models_assets.get('rf_features', [])}")

st.divider()


# =============================================================================
# SIDEBAR
# =============================================================================
st.sidebar.header("Filters")
content_options = ["All"] + sorted([x for x in df["content_type"].dropna().astype(str).unique().tolist() if x != ""])
selected_content = st.sidebar.selectbox("Content type", content_options, index=0)

if "source" in df.columns:
    source_options = ["All"] + sorted([x for x in df["source"].dropna().astype(str).unique().tolist() if x != ""])
else:
    source_options = ["All"]
selected_source = st.sidebar.selectbox("Source", source_options, index=0)

language_options = ["All"] + sorted([x for x in df.get("original_language", pd.Series(dtype=str)).dropna().astype(str).unique().tolist() if x != ""])
selected_languages = st.sidebar.multiselect("Languages", language_options, default=["All"])

all_detected_topics = sorted({topic for topics in df["detected_topics_auto"] for topic in topics})
selected_topics = st.sidebar.multiselect("Detected themes", all_detected_topics, default=[])

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider("Release year range", min_year, max_year, (min_year, max_year))
else:
    year_range = None

st.sidebar.divider()
sidebar_title_options = sorted(df["title"].dropna().astype(str).unique().tolist())
sidebar_selected_title = st.sidebar.selectbox(
    "Quick title search",
    options=sidebar_title_options,
    index=None,
    placeholder="Start typing...",
)


# =============================================================================
# APPLY FILTERS
# =============================================================================
filtered_df = df.copy()
if selected_content != "All":
    filtered_df = filtered_df[filtered_df["content_type"] == selected_content]
if selected_source != "All" and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"] == selected_source]
if "All" not in selected_languages and selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]
if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")]
if selected_topics:
    filtered_df = filtered_df[filtered_df["detected_topics_auto"].apply(lambda x: any(topic in x for topic in selected_topics))]


# =============================================================================
# QUICK TITLE SNAPSHOT
# =============================================================================
if sidebar_selected_title:
    sidebar_row = df[df["title"] == sidebar_selected_title].head(1)
    if not sidebar_row.empty:
        row = sidebar_row.iloc[0]
        st.sidebar.markdown("#### Title snapshot")
        st.sidebar.markdown(
            f"**Type:** {row.get('content_type', 'N/A')}  \n"
            f"**Year:** {int(row['release_year']) if pd.notna(row.get('release_year')) else 'N/A'}  \n"
            f"**Language:** {row.get('original_language', 'N/A')}"
        )
        st.sidebar.metric("Visibility", metric_str(row.get("visibility_score", np.nan)))
        st.sidebar.metric("Engagement", metric_str(row.get("engagement_score", np.nan)))
        st.sidebar.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))
        if pd.notna(row.get("predicted_business_value", np.nan)):
            st.sidebar.metric("Predicted BV", metric_str(row.get("predicted_business_value", np.nan)))
        if "cluster_ev" in row.index:
            st.sidebar.metric("Cluster EV", str(row.get("cluster_ev")))
        if "cluster_final" in row.index:
            st.sidebar.metric("Cluster Final", str(row.get("cluster_final")))
        active_topics = get_active_topics_from_row(row)
        if not active_topics:
            active_topics = row.get("detected_topics_auto", [])
        st.sidebar.caption("Themes: " + (", ".join(active_topics) if active_topics else "Not detected"))


# =============================================================================
# KPI STRIP
# =============================================================================
st.markdown(f"### Filtered catalog: {filtered_df.shape[0]:,} titles")

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total titles", f"{len(filtered_df):,}")
k2.metric("Avg popularity", metric_str(filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan))
k3.metric("Avg business value", metric_str(filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan))
k4.metric("Avg predicted BV", metric_str(filtered_df["predicted_business_value"].mean() if "predicted_business_value" in filtered_df.columns else np.nan))

k5, k6, k7, k8 = st.columns(4)
k5.metric("Avg visibility", metric_str(filtered_df["visibility_score"].mean() if "visibility_score" in filtered_df.columns else np.nan))
k6.metric("Avg engagement", metric_str(filtered_df["engagement_score"].mean() if "engagement_score" in filtered_df.columns else np.nan))
k7.metric("Movies", f"{int((filtered_df['content_type'].astype(str).str.lower() == 'movie').sum()):,}" if "content_type" in filtered_df.columns else "N/A")
k8.metric("Series / TV", f"{int((filtered_df['content_type'].astype(str).str.lower().isin(['tv', 'series'])).sum()):,}" if "content_type" in filtered_df.columns else "N/A")

st.divider()


# =============================================================================
# TABS
# =============================================================================
tab1, tab2, tab3, tab4, tab5, tab6, tab7 = st.tabs([
    "📊 Executive Overview",
    "🎯 Marketing Performance",
    "🧠 Audience & Themes",
    "🧩 Clustering Insights",
    "🔎 Similar Titles Finder",
    "📝 Synopsis Opportunity Lab",
    "🤖 Model Lab",
])


# =============================================================================
# TAB 1 – EXECUTIVE OVERVIEW
# =============================================================================
with tab1:
    st.subheader("Executive overview")
    c1, c2 = st.columns(2)
    with c1:
        if "content_type" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["content_type"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_1)
            ax.set_title("Titles by content type", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)
    with c2:
        if "source" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["source"].value_counts().head(10)
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_2)
            ax.set_title("Top sources", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)
    with c3:
        if "business_value_score" in filtered_df.columns and filtered_df["business_value_score"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["business_value_score"].dropna(), bins=30, color=ACCENT_5)
            ax.set_title("Business value distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)
    with c4:
        if "predicted_business_value" in filtered_df.columns and filtered_df["predicted_business_value"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["predicted_business_value"].dropna(), bins=30, color=ACCENT_3)
            ax.set_title("Predicted business value distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)

    if "release_year" in filtered_df.columns and filtered_df["release_year"].notna().any():
        yearly = filtered_df["release_year"].dropna().astype(int).value_counts().sort_index()
        fig, ax = make_dark_fig((12, 4))
        ax.plot(yearly.index, yearly.values)
        ax.set_title("Catalog trend by release year", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        st.pyplot(fig)
        plt.close(fig)


# =============================================================================
# TAB 2 – MARKETING PERFORMANCE
# =============================================================================
with tab2:
    st.subheader("Marketing performance analysis")
    metric_map = {
        "Visibility": "visibility_score",
        "Engagement": "engagement_score",
        "Business Value": "business_value_score",
        "Predicted BV": "predicted_business_value",
        "Popularity": "popularity",
        "Audience Reception": "audience_reception_score",
    }
    available = [k for k, v in metric_map.items() if v in filtered_df.columns]
    selected_metric_label = st.radio("Highlighted KPI", available, horizontal=True, index=available.index("Predicted BV") if "Predicted BV" in available else 0)
    highlight_metric = metric_map[selected_metric_label]

    c1, c2 = st.columns(2)
    with c1:
        top_df = filtered_df[["title", highlight_metric]].dropna().sort_values(by=highlight_metric, ascending=False).head(10)
        if not top_df.empty:
            fig, ax = make_dark_fig((8, 5))
            ax.barh(top_df["title"][::-1], top_df[highlight_metric][::-1], color=ACCENT_1)
            ax.set_title(f"Top 10 by {selected_metric_label}", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)
    with c2:
        if all(c in filtered_df.columns for c in ["popularity", "predicted_business_value"]):
            sample = filtered_df[["popularity", "predicted_business_value"]].dropna()
            if not sample.empty:
                sample = sample.sample(min(len(sample), 5000), random_state=42)
                fig, ax = make_dark_fig((8, 5))
                ax.scatter(sample["popularity"], sample["predicted_business_value"], alpha=0.35)
                ax.set_title("Popularity vs Predicted Business Value", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                st.pyplot(fig)
                plt.close(fig)


# =============================================================================
# TAB 3 – AUDIENCE & THEMES
# =============================================================================
with tab3:
    st.subheader("Audience and thematic signals")
    c1, c2 = st.columns(2)
    with c1:
        topic_counts = {}
        for topic_list in filtered_df["detected_topics_auto"]:
            for topic in topic_list:
                topic_counts[topic] = topic_counts.get(topic, 0) + 1
        if topic_counts:
            topic_series = pd.Series(topic_counts).sort_values(ascending=False).head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.barh(topic_series.index[::-1], topic_series.values[::-1], color=ACCENT_3)
            ax.set_title("Top detected themes", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)
    with c2:
        if "genre_list" in filtered_df.columns:
            exploded = filtered_df.assign(genre_item=filtered_df["genre_list"]).explode("genre_item")
            exploded = exploded[exploded["genre_item"].notna() & (exploded["genre_item"] != "")]
            if not exploded.empty:
                top_genres = exploded["genre_item"].value_counts().head(12)
                fig, ax = make_dark_fig((8, 5))
                ax.barh(top_genres.index[::-1], top_genres.values[::-1], color=ACCENT_5)
                ax.set_title("Top genres", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                st.pyplot(fig)
                plt.close(fig)


# =============================================================================
# TAB 4 – CLUSTERING INSIGHTS
# =============================================================================
with tab4:
    st.subheader("Clustering insights")
    if "cluster_ev" in filtered_df.columns:
        c1, c2 = st.columns(2)
        with c1:
            counts = filtered_df["cluster_ev"].value_counts().sort_index()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index.astype(str), counts.values, color=ACCENT_2)
            ax.set_title("KMeans Engage/Visibility clusters", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)
        with c2:
            cluster_summary_cols = [c for c in ["cluster_ev", "visibility_score", "engagement_score", "business_value_score", "predicted_business_value"] if c in filtered_df.columns]
            if len(cluster_summary_cols) >= 2:
                cluster_summary = filtered_df[cluster_summary_cols].groupby("cluster_ev").mean(numeric_only=True).round(2)
                st.dataframe(cluster_summary, use_container_width=True)
    else:
        st.info("Engage/Visibility KMeans model could not be applied with the available inputs.")

    if "cluster_final" in filtered_df.columns:
        st.markdown("### Final segmentation")
        counts = filtered_df["cluster_final"].value_counts().sort_index()
        fig, ax = make_dark_fig((7, 4))
        ax.bar(counts.index.astype(str), counts.values, color=ACCENT_4)
        ax.set_title("Final KMeans clusters", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        st.pyplot(fig)
        plt.close(fig)

        final_summary_cols = [c for c in ["cluster_final", "visibility_score", "engagement_score", "business_value_score", "predicted_business_value"] if c in filtered_df.columns]
        if len(final_summary_cols) >= 2:
            final_summary = filtered_df[final_summary_cols].groupby("cluster_final").mean(numeric_only=True).round(2)
            st.dataframe(final_summary, use_container_width=True)
    else:
        st.info("Final KMeans model could not be applied with the available inputs.")


# =============================================================================
# TAB 5 – SIMILAR TITLES FINDER
# =============================================================================
with tab5:
    st.subheader("Similar titles finder")
    available_titles = sorted(filtered_df["title"].dropna().astype(str).unique().tolist())
    if not available_titles:
        st.warning("No titles available under the current filters.")
    else:
        selected_title = st.selectbox("Start typing and select a title", available_titles, index=None, placeholder="Type a movie or series title...")
        if selected_title:
            row = filtered_df[filtered_df["title"] == selected_title].head(1).iloc[0]
            i1, i2, i3, i4 = st.columns(4)
            i1.metric("Type", row.get("content_type", "N/A"))
            i2.metric("Year", int(row["release_year"]) if pd.notna(row.get("release_year")) else "N/A")
            i3.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))
            i4.metric("Predicted BV", metric_str(row.get("predicted_business_value", np.nan)))

            similar_df = get_similar_titles(df_similarity, sim_matrix, selected_title, top_n=10)
            if similar_df.empty:
                st.info("No similar titles found.")
            else:
                fig, ax = make_dark_fig((9, 5))
                plot_df = similar_df.head(8)
                ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_1)
                ax.set_title("Top similarity", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                st.pyplot(fig)
                plt.close(fig)
                st.dataframe(similar_df.round(3), use_container_width=True)
                csv_similar = similar_df.to_csv(index=False).encode("utf-8-sig")
                st.download_button("Download similar titles", csv_similar, "similar_titles_results.csv", "text/csv")


# =============================================================================
# TAB 6 – SYNOPSIS OPPORTUNITY LAB
# =============================================================================
with tab6:
    st.subheader("Synopsis opportunity lab")
    st.markdown(
        "Paste a new synopsis and the app will detect themes, find the closest comparable titles, and estimate a business value from those comparables."
    )
    user_type = st.selectbox("Content type", ["All", "movie", "tv"], index=0)
    user_synopsis = st.text_area(
        "Paste the synopsis",
        height=180,
        placeholder="Example: A young journalist uncovers a corruption network while trying to protect her family and save her career...",
    )
    top_n = st.slider("Number of comparables", 3, 10, 5)

    if st.button("Analyze synopsis"):
        if not user_synopsis.strip():
            st.warning("Please paste a synopsis first.")
        else:
            detected_topics = detect_topics_from_synopsis(user_synopsis)
            synopsis_results = get_top_similar_titles_from_synopsis(user_synopsis, user_type, df_similarity, top_n=top_n)
            estimated_bv = estimate_synopsis_business_value(synopsis_results)
            estimated_band = business_value_band(estimated_bv)

            st.session_state["synopsis_results"] = synopsis_results
            st.session_state["user_synopsis_topics"] = detected_topics
            st.session_state["estimated_bv"] = estimated_bv
            st.session_state["estimated_band"] = estimated_band
            st.session_state["user_synopsis"] = user_synopsis
            st.session_state["user_type"] = user_type

    if "synopsis_results" in st.session_state:
        synopsis_results = st.session_state["synopsis_results"]
        detected_topics = st.session_state.get("user_synopsis_topics", [])
        estimated_bv = st.session_state.get("estimated_bv", np.nan)
        estimated_band = st.session_state.get("estimated_band", "Unknown")

        a1, a2, a3 = st.columns(3)
        a1.metric("Detected themes", f"{len(detected_topics)}")
        a2.metric("Estimated Business Value", metric_str(estimated_bv))
        a3.metric("Commercial assessment", estimated_band)

        st.markdown("### Detected themes")
        st.write(", ".join(detected_topics) if detected_topics else "No clear themes detected.")

        st.markdown("### Closest titles")
        if synopsis_results.empty:
            st.info("No comparable titles were found.")
        else:
            fig, ax = make_dark_fig((9, 5))
            plot_df = synopsis_results.head(8)
            ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_2)
            ax.set_title("Top comparable titles by similarity", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            st.pyplot(fig)
            plt.close(fig)
            st.dataframe(synopsis_results.round(3), use_container_width=True)

            if pd.notna(estimated_bv):
                if estimated_bv >= 75:
                    st.success("This synopsis resembles titles with strong commercial potential.")
                elif estimated_bv >= 55:
                    st.info("This synopsis resembles titles with medium-high potential.")
                elif estimated_bv >= 40:
                    st.warning("This synopsis resembles titles with moderate potential.")
                else:
                    st.error("This synopsis resembles titles with lower estimated commercial value in the current dataset.")

            export_df = synopsis_results.copy()
            export_df.insert(0, "input_synopsis", st.session_state.get("user_synopsis", ""))
            export_df.insert(1, "input_content_type", st.session_state.get("user_type", "All"))
            export_df.insert(2, "detected_topics", ", ".join(detected_topics))
            export_df.insert(3, "estimated_business_value", estimated_bv)
            export_df.insert(4, "estimated_business_value_band", estimated_band)
            csv_synopsis = export_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button("Download synopsis analysis", csv_synopsis, "synopsis_opportunity_results.csv", "text/csv")


# =============================================================================
# TAB 7 – MODEL LAB
# =============================================================================
with tab7:
    st.subheader("Model lab")

    st.markdown("### Random Forest feature importance")
    fi = feature_importance_frame(models_assets.get("rf_model"), models_assets.get("rf_features", RF_FEATURES))
    if fi.empty:
        st.info("Feature importances are not available. The model may not be loaded or may not expose them.")
    else:
        fig, ax = make_dark_fig((10, 5))
        plot_df = fi.head(10)
        ax.barh(plot_df["feature"][::-1], plot_df["importance"][::-1], color=ACCENT_3)
        ax.set_title("Top feature importances", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        st.pyplot(fig)
        plt.close(fig)
        st.dataframe(fi, use_container_width=True)

    st.markdown("### Existing titles: actual vs predicted")
    if all(c in filtered_df.columns for c in ["business_value_score", "predicted_business_value"]):
        comp = filtered_df[["business_value_score", "predicted_business_value"]].dropna()
        if not comp.empty:
            comp = comp.sample(min(len(comp), 3000), random_state=42)
            fig, ax = make_dark_fig((8, 6))
            ax.scatter(comp["business_value_score"], comp["predicted_business_value"], alpha=0.35)
            mn = min(comp.min())
            mx = max(comp.max())
            ax.plot([mn, mx], [mn, mx], linestyle="--")
            ax.set_title("Predicted vs Actual Business Value", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Actual", color=TEXT)
            ax.set_ylabel("Predicted", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    build_manual_rf_input(models_assets)
'''

app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    webbrowser.open(url)
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()


Starting Streamlit on http://127.0.0.1:56488 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:56488

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:56488


2026-08-07 01:49:44.464 Pandas DataFrame hash failed. Falling back to pickling the object.
Traceback (most recent call last):
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/streamlit/runtime/caching/hashing.py", line 454, in _to_bytes
    values_hash_bytes = self.to_bytes(hash_pandas_object(df_obj))
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 174, in hash_pandas_object
    h = combine_hash_arrays(hashes, num_items)
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 72, in combine_hash_arrays
    for i, a in enumerate(arrays):
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 173, in <genexpr>
    hashes = (x for x in _hashes)
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 154, in <genexpr>
    hash_array(series._values, encoding, hash_key, categorize)
  File 

# FINAL 

In [ ]:
import socket
import subprocess
import time
import urllib.request
import sys
import webbrowser
from pathlib import Path


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def wait_for_server(url, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(0.5)
    return False

streamlit_code = r'''
import re
import ast
from pathlib import Path

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Streaming Marketing Intelligence",
    page_icon="🎬",
    layout="wide"
)

CARD_BG = "#1E293B"
FIG_BG = "#0F172A"
TEXT = "white"
MUTED = "#94A3B8"
ACCENT_1 = "#6366F1"
ACCENT_2 = "#10B981"
ACCENT_3 = "#F59E0B"
ACCENT_4 = "#EF4444"
ACCENT_5 = "#22C55E"

DEFAULT_DATASET_CANDIDATES = [
    "DATA/PROCESSED/all_streaming_titles_enriched.csv",
]

TOPIC_KEYWORDS = {
    "lgbtq": [
        "gay", "lesbian", "lgbt", "trans", "queer", "bisexual",
        "nonbinary", "coming out", "drag", "identity"
    ],

    "politics": [
        "politics", "president", "government", "election", "policy",
        "senate", "congress", "campaign", "minister", "dictator",
        "democracy", "corruption", "power", "state"
    ],

    "climate_environment": [
        "climate", "environment", "global warming", "pollution",
        "ecology", "sustainability", "nature", "forest", "wildlife",
        "environmental disaster", "carbon", "climate crisis"
    ],

    "war_military": [
        "war", "battle", "army", "soldier", "military", "conflict",
        "weapon", "navy", "air force", "commander", "resistance",
        "invasion", "combat", "veteran"
    ],

    "family": [
        "family", "mother", "father", "parent", "children", "child",
        "home", "siblings", "brother", "sister", "marriage",
        "divorce", "parenthood", "relatives"
    ],

    "crime": [
        "crime", "murder", "police", "detective", "investigation",
        "killer", "gang", "mafia", "cartel", "robbery", "heist",
        "forensics", "prison", "criminal", "underworld"
    ],

    "policial": [
        "police", "cop", "officer", "detective", "agent", "rookie",
        "investigation", "investigator", "inspector", "sergeant",
        "lieutenant", "captain", "chief", "precinct", "partner",
        "case", "crime scene", "forensics", "suspect", "witness",
        "interrogation", "undercover", "surveillance", "homicide",
        "manhunt", "law enforcement", "special unit", "task force",
        "fbi", "cia", "narcotics", "patrol", "unit"
    ],

    "romance": [
        "love", "romance", "relationship", "couple", "passion",
        "heartbreak", "dating", "affair", "wedding", "breakup",
        "soulmate", "jealousy"
    ],

    "technology": [
        "technology", "ai", "artificial intelligence", "robot",
        "future", "cyber", "computer", "hacker", "internet",
        "virtual reality", "machine", "automation", "surveillance"
    ],

    "mental_health": [
        "depression", "anxiety", "trauma", "therapy", "mental",
        "psychological", "stress", "grief", "addiction", "bipolar",
        "schizophrenia", "panic", "healing", "psychiatric"
    ],

    "coming_of_age": [
        "teen", "adolescent", "growing up", "school", "youth",
        "friendship", "identity", "first love", "high school",
        "college", "self-discovery", "maturity"
    ],

    "social_issues": [
        "racism", "inequality", "poverty", "discrimination", "justice",
        "migration", "sexism", "class", "homophobia", "xenophobia",
        "oppression", "human rights", "activism"
    ],

    "fantasy_supernatural": [
        "magic", "witch", "dragon", "supernatural", "monster",
        "curse", "fantasy", "sorcery", "wizard", "demon", "ghost",
        "prophecy", "kingdom", "spell"
    ],

    "science_fiction": [
        "space", "alien", "spaceship", "future", "planet",
        "time travel", "parallel universe", "android", "mutation",
        "dystopia", "utopia", "interstellar", "extraterrestrial"
    ],

    "horror": [
        "horror", "fear", "haunted", "ghost", "possession",
        "slasher", "evil", "demon", "nightmare", "blood",
        "terror", "zombie", "paranormal"
    ],

    "thriller": [
        "thriller", "suspense", "mystery", "conspiracy", "chase",
        "secret", "obsession", "danger", "kidnapping", "betrayal",
        "tension", "survival"
    ],

    "action_adventure": [
        "action", "adventure", "hero", "mission", "explosion",
        "fight", "chase", "survival", "escape", "journey",
        "quest", "mercenary"
    ],

    "historical": [
        "history", "historical", "period drama", "king", "queen",
        "empire", "revolution", "civilization", "medieval",
        "ancient", "biographical", "royalty"
    ],

    "biography": [
        "biography", "biopic", "true story", "real life", "famous",
        "artist", "scientist", "politician", "athlete", "inventor"
    ],

    "sports": [
        "sport", "football", "soccer", "basketball", "baseball",
        "tennis", "boxing", "fighter", "competition", "tournament",
        "coach", "team", "championship"
    ],

    "music_performance": [
        "music", "band", "singer", "concert", "performance",
        "musician", "song", "dance", "ballet", "opera",
        "stage", "show business"
    ],

    "comedy": [
        "comedy", "funny", "humor", "satire", "parody",
        "awkward", "absurd", "joke", "laugh", "misunderstanding"
    ],

    "drama": [
        "drama", "emotional", "conflict", "sacrifice", "loss",
        "betrayal", "redemption", "personal struggle", "intense"
    ],

    "mystery": [
        "mystery", "secret", "clue", "disappearance", "puzzle",
        "unknown", "hidden truth", "unsolved", "investigation"
    ],

    "survival_disaster": [
        "survival", "disaster", "earthquake", "tsunami", "fire",
        "shipwreck", "plane crash", "apocalypse", "epidemic",
        "outbreak", "catastrophe"
    ],

    "religion_spirituality": [
        "religion", "faith", "god", "church", "priest", "spiritual",
        "belief", "miracle", "sacred", "ritual", "afterlife"
    ],

    "animation_family": [
        "animation", "animated", "family-friendly", "kids",
        "talking animals", "fairy tale", "adventure for children"
    ],

    "friendship": [
        "friendship", "best friends", "companionship", "bond",
        "loyalty", "group of friends", "reunion"
    ],

    "revenge": [
        "revenge", "vengeance", "payback", "betrayal", "justice",
        "retaliation", "avenger"
    ],

    "road_trip_journey": [
        "road trip", "journey", "travel", "on the road", "escape",
        "self-discovery", "destination"
    ],

    "school_university": [
        "school", "teacher", "student", "classroom", "university",
        "college", "campus", "exam", "bullying", "graduation"
    ],

    "work_business": [
        "work", "office", "career", "boss", "company", "business",
        "corporate", "startup", "ambition", "promotion", "colleague"
    ]
}


# ──────────────────────────────────────────────────────────────────────────────
# STYLE
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
.block-container {
    padding-top: 1.2rem;
    padding-bottom: 2rem;
}
div[data-testid="stMetric"] {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    padding: 14px 16px;
    border-radius: 16px;
}
div[data-testid="stDataFrame"] {
    border-radius: 12px;
}
.small-card {
    background-color: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 14px;
    padding: 12px 14px;
    margin-top: 8px;
    margin-bottom: 8px;
}
.objective-card {
    background: linear-gradient(135deg, #0F172A 0%, #111827 100%);
    border: 1px solid #1E293B;
    border-radius: 18px;
    padding: 18px 20px;
    margin-bottom: 16px;
}
.section-title {
    font-size: 1.1rem;
    font-weight: 700;
    color: white;
    margin-bottom: 0.5rem;
}
.muted {
    color: #94A3B8;
}
</style>
""", unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9áéíóúñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def metric_str(value, decimals=2):
    return f"{value:.{decimals}f}" if pd.notna(value) else "N/A"


def parse_people(value):
    """Return clean person names from an IMDb pipe-separated field."""
    if pd.isna(value):
        return []

    return [
        person.strip()
        for person in str(value).split("|")
        if person.strip()
    ]


def unique_people_options(series):
    """Build sorted type-ahead options from a pipe-separated people column."""
    people = {
        person
        for value in series.fillna("")
        for person in parse_people(value)
    }
    return sorted(people, key=str.casefold)


def row_has_person(value, selected_person):
    """Check exact person membership inside a pipe-separated field."""
    if not selected_person:
        return True
    return selected_person in parse_people(value)


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def parse_genres(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [g.strip() for g in text.split(",") if g.strip()]


def detect_topics_from_synopsis(text: str):
    text_clean = clean_text(text)
    found_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword.lower() in text_clean for keyword in keywords):
            found_topics.append(topic)

    if not found_topics:
        found_topics.append("General / No clear topic")

    return found_topics


def get_active_topics_from_row(row):
    topics = []

    if "top_topics" in row.index and pd.notna(row.get("top_topics")) and str(row.get("top_topics")).strip():
        raw = str(row.get("top_topics"))
        if "|" in raw:
            topics.extend([x.strip() for x in raw.split("|") if x.strip()])
        elif "," in raw:
            topics.extend([x.strip() for x in raw.split(",") if x.strip()])
        else:
            topics.append(raw.strip())

    topic_like_cols = [c for c in row.index if c.startswith("topic_") and c != "topic_diversity_score"]
    for col in topic_like_cols:
        try:
            if float(row.get(col, 0)) == 1:
                topics.append(col.replace("topic_", "").replace("_", " ").title())
        except Exception:
            continue

    topics = [t for t in topics if t]
    return sorted(list(set(topics)))


def make_dark_fig(figsize=(8, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(FIG_BG)
    ax.set_facecolor(CARD_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TEXT)
    return fig, ax


def normalize_0_100(series):
    series = pd.to_numeric(series, errors="coerce")
    valid = series.dropna()
    if valid.empty:
        return pd.Series(np.nan, index=series.index)
    min_v = valid.min()
    max_v = valid.max()
    if min_v == max_v:
        return pd.Series(50.0, index=series.index)
    return ((series - min_v) / (max_v - min_v)) * 100


def ensure_business_value_score(df):
    if "business_value_score" in df.columns and df["business_value_score"].notna().any():
        return df

    pop_norm = normalize_0_100(df["popularity"]) if "popularity" in df.columns else pd.Series(np.nan, index=df.index)
    vote_avg_norm = normalize_0_100(df["vote_average"]) if "vote_average" in df.columns else pd.Series(np.nan, index=df.index)
    vote_count_norm = normalize_0_100(df["vote_count"]) if "vote_count" in df.columns else pd.Series(np.nan, index=df.index)
    visibility = normalize_0_100(df["visibility_score"]) if "visibility_score" in df.columns else pd.Series(np.nan, index=df.index)
    engagement = normalize_0_100(df["engagement_score"]) if "engagement_score" in df.columns else pd.Series(np.nan, index=df.index)
    reception = normalize_0_100(df["audience_reception_score"]) if "audience_reception_score" in df.columns else pd.Series(np.nan, index=df.index)

    pieces = pd.concat(
        [pop_norm, vote_avg_norm, vote_count_norm, visibility, engagement, reception],
        axis=1
    )
    pieces.columns = ["pop", "vote_avg", "vote_count", "visibility", "engagement", "reception"]

    weights = {
        "pop": 0.25,
        "vote_avg": 0.20,
        "vote_count": 0.15,
        "visibility": 0.15,
        "engagement": 0.10,
        "reception": 0.15,
    }

    weighted = sum(pieces[col].fillna(pieces.mean(axis=1)) * w for col, w in weights.items())
    df["business_value_score"] = weighted.round(2)

    return df


def estimate_synopsis_business_value(similar_df):
    if similar_df.empty:
        return np.nan

    score_col = None
    if "predicted_business_value" in similar_df.columns and similar_df["predicted_business_value"].notna().any():
        score_col = "predicted_business_value"
    elif "business_value_score" in similar_df.columns and similar_df["business_value_score"].notna().any():
        score_col = "business_value_score"

    if score_col is None:
        return np.nan

    work = similar_df[[score_col, "similarity_score"]].copy()
    work = work.dropna()

    if work.empty:
        return np.nan

    weights = work["similarity_score"].clip(lower=0.001)
    value = np.average(work[score_col], weights=weights)
    return round(float(value), 2)


def business_value_band(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 75:
        return "High commercial potential"
    if score >= 55:
        return "Medium-high potential"
    if score >= 40:
        return "Moderate potential"
    return "Lower potential"


def dataset_health_text(df):
    parts = []
    parts.append(f"{len(df):,} titles")
    if "content_type" in df.columns:
        parts.append(f"{df['content_type'].nunique()} content types")
    if "source" in df.columns:
        parts.append(f"{df['source'].nunique()} sources")
    if "release_year" in df.columns and df["release_year"].notna().any():
        parts.append(
            f"years {int(df['release_year'].min())}–{int(df['release_year'].max())}"
        )
    return " · ".join(parts)


def choose_dataset_file():
    for candidate in DEFAULT_DATASET_CANDIDATES:
        if Path(candidate).exists():
            return candidate
    return None


# ──────────────────────────────────────────────────────────────────────────────
# DATA
# ──────────────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    dataset_path = choose_dataset_file()
    if dataset_path is None:
        raise FileNotFoundError(
            "No dataset found. Please place one of these files in the app folder: "
            "DATA/PROCESSED/all_streaming_titles_enriched.csv"
        )

    df = pd.read_csv(dataset_path)

    # Core cleanup
    if "release_date" in df.columns:
        df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

    numeric_cols = [
        "popularity", "vote_average", "vote_count",
        "visibility_score", "engagement_score",
        "business_value_score", "predicted_business_value",
        "topic_diversity_score", "runtime_final", "cluster",
        "audience_reception_score", "freshness_score",
        "release_year"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    text_cols = [
        "title", "overview", "overview_clean", "genre_names",
        "content_type", "source", "original_language",
        "marketing_segment", "cluster_label", "production_companies",
        "network", "status", "web_channel", "top_topics",
        "imdb_cast", "imdb_directors", "imdb_cinematographers"
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    if "genre_names" in df.columns:
        df["genre_list"] = df["genre_names"].apply(parse_genres)
    else:
        df["genre_list"] = [[] for _ in range(len(df))]

    if "overview_clean" not in df.columns:
        df["overview_clean"] = df["overview"].fillna("").apply(clean_text)
    else:
        df["overview_clean"] = df["overview_clean"].fillna("").apply(clean_text)

    if "title" not in df.columns:
        df["title"] = "Untitled"

    if "content_type" not in df.columns:
        df["content_type"] = "unknown"

    df["title_key"] = df["title"].fillna("").astype(str).str.lower().str.strip()
    df["detected_topics_auto"] = df["overview"].fillna("").apply(detect_topics_from_synopsis)
    df["detected_topics_text"] = df["detected_topics_auto"].apply(lambda x: ", ".join(x) if x else "")

    df = ensure_business_value_score(df)

    if "predicted_business_value" not in df.columns:
        df["predicted_business_value"] = np.nan

    if "cluster_label" not in df.columns:
        df["cluster_label"] = ""

    if "marketing_segment" not in df.columns:
        df["marketing_segment"] = ""

    return df, dataset_path


@st.cache_data
def prepare_similarity(df):
    work_df = df.copy()

    work_df["topics_text"] = work_df.apply(
        lambda row: " ".join(get_active_topics_from_row(row)) if get_active_topics_from_row(row) else row.get("detected_topics_text", ""),
        axis=1
    )

    genre_text = work_df["genre_list"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
    work_df["similarity_text"] = (
        work_df["title"].fillna("") + " "
        + genre_text.fillna("") + " "
        + work_df["overview_clean"].fillna("") + " "
        + work_df["topics_text"].fillna("") + " "
        + work_df["original_language"].fillna("")
    ).str.lower()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=6000,
        ngram_range=(1, 2)
    )
    matrix = vectorizer.fit_transform(work_df["similarity_text"])

    return work_df, matrix


def get_similar_titles(df_similarity, matrix, selected_title, top_n=10):
    matches = df_similarity.index[df_similarity["title"] == selected_title].tolist()
    if not matches:
        return pd.DataFrame()

    idx = matches[0]
    sims = cosine_similarity(matrix[idx], matrix).flatten()

    sim_df = df_similarity.copy()
    sim_df["similarity_score"] = sims
    sim_df = sim_df[sim_df.index != idx].copy()

    sort_cols = ["similarity_score", "business_value_score", "vote_average", "popularity"]
    sort_cols = [c for c in sort_cols if c in sim_df.columns]

    rank_cols = [
        "title", "content_type", "genre_names", "release_year",
        "vote_average", "popularity", "visibility_score",
        "engagement_score", "business_value_score",
        "predicted_business_value", "marketing_segment",
        "cluster_label", "detected_topics_text", "similarity_score", "overview"
    ]
    existing_cols = [c for c in rank_cols if c in sim_df.columns]

    return sim_df.sort_values(
        by=sort_cols,
        ascending=[False] * len(sort_cols)
    )[existing_cols].head(top_n)


def get_top_similar_titles_from_synopsis(user_synopsis, content_type, df_similarity, matrix, top_n=5):
    working = df_similarity.copy()

    if content_type and content_type.lower() != "all" and "content_type" in working.columns:
        working = working[
            working["content_type"].astype(str).str.lower() == content_type.lower()
        ].copy()

    if working.empty:
        return pd.DataFrame()

    user_text = clean_text(user_synopsis)
    user_topics = set(detect_topics_from_synopsis(user_synopsis))

    user_vec = matrix.__class__(matrix.shape[0])  # placeholder not used directly

    # Need to rebuild on filtered subset from existing vectorizer behavior:
    # simplest safe path: refit a local vectorizer on filtered rows
    local_vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=6000,
        ngram_range=(1, 2)
    )
    local_matrix = local_vectorizer.fit_transform(working["similarity_text"])
    user_vec = local_vectorizer.transform([user_text])
    text_sims = cosine_similarity(user_vec, local_matrix).flatten()

    working["text_similarity"] = text_sims

    def topic_overlap_score(item_topics_text):
        item_topics = set([x.strip() for x in str(item_topics_text).split(",") if x.strip()])
        if not user_topics or not item_topics:
            return 0
        return len(user_topics.intersection(item_topics)) / max(len(user_topics), 1)

    working["topic_similarity"] = working["detected_topics_text"].apply(topic_overlap_score)

    working["similarity_score"] = (
        0.80 * working["text_similarity"] +
        0.20 * working["topic_similarity"]
    )

    cols_to_show = [
        col for col in [
            "title",
            "content_type",
            "genre_names",
            "original_language",
            "release_year",
            "overview",
            "business_value_score",
            "predicted_business_value",
            "marketing_segment",
            "cluster_label",
            "detected_topics_text",
            "text_similarity",
            "topic_similarity",
            "similarity_score"
        ] if col in working.columns
    ]

    result = working.sort_values(
        by=["similarity_score", "business_value_score"],
        ascending=[False, False]
    ).head(top_n)

    return result[cols_to_show]


def get_rank(df_all, title, col):
    if col not in df_all.columns:
        return None, None

    rank_df = df_all[["title", col]].dropna().sort_values(by=col, ascending=False).reset_index(drop=True)
    rank_df.index = rank_df.index + 1
    matches = rank_df[rank_df["title"] == title]

    if matches.empty:
        return None, len(rank_df)

    return int(matches.index[0]), len(rank_df)


# ──────────────────────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────────────────────
df, dataset_path = load_data()
df_similarity, sim_matrix = prepare_similarity(df)


# ──────────────────────────────────────────────────────────────────────────────
# HEADER
# ──────────────────────────────────────────────────────────────────────────────
st.markdown("## 🎬 Streaming Marketing Intelligence Dashboard")

st.markdown(f"""
<div class="objective-card">
    <div class="section-title">Marketing Objective</div>
    <div class="muted">
        Support content marketing and acquisition decisions by identifying which titles show stronger commercial potential,
        what themes are most attractive, and which existing titles can serve as relevant comparables for a new idea or synopsis.
    </div>
    <br>
    <div class="muted">
        <strong>Primary use cases:</strong> content benchmarking · similarity search · idea validation · business value estimation · marketing storytelling
    </div>
</div>
""", unsafe_allow_html=True)

st.caption(f"Dataset loaded: `{dataset_path}` · {dataset_health_text(df)}")
st.divider()


# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────────────────────────────────────
st.sidebar.header("Filters")

content_options_real = sorted([x for x in df["content_type"].dropna().astype(str).unique().tolist() if x != ""])
content_options = ["All"] + content_options_real
selected_content = st.sidebar.selectbox("Content type", options=content_options, index=0)

source_options_real = sorted([x for x in df["source"].dropna().astype(str).unique().tolist() if x != ""]) if "source" in df.columns else []
source_options = ["All"] + source_options_real
selected_source = st.sidebar.selectbox("Source", options=source_options, index=0)

language_options_real = sorted([x for x in df["original_language"].dropna().astype(str).unique().tolist() if x != ""]) if "original_language" in df.columns else []
selected_languages = st.sidebar.multiselect(
    "Languages",
    options=["All"] + language_options_real,
    default=["All"]
)

all_detected_topics = sorted({
    topic
    for topics in df["detected_topics_auto"]
    for topic in topics
})
selected_topics = st.sidebar.multiselect(
    "Detected themes",
    options=all_detected_topics,
    default=[]
)

if "release_year" in df.columns and df["release_year"].notna().any():
    min_year = int(df["release_year"].dropna().min())
    max_year = int(df["release_year"].dropna().max())
    year_range = st.sidebar.slider(
        "Release year range",
        min_value=min_year,
        max_value=max_year,
        value=(min_year, max_year)
    )
else:
    year_range = None

st.sidebar.divider()
st.sidebar.markdown("### Quick title search")

sidebar_title_options = sorted(df["title"].dropna().astype(str).unique().tolist())
sidebar_selected_title = st.sidebar.selectbox(
    "Search a title",
    options=sidebar_title_options,
    index=None,
    placeholder="Start typing..."
)

st.sidebar.divider()
st.sidebar.markdown("### Talent filters")

cast_options = (
    unique_people_options(df["imdb_cast"])
    if "imdb_cast" in df.columns
    else []
)
selected_cast = st.sidebar.selectbox(
    "Cast member",
    options=cast_options,
    index=None,
    placeholder="Start typing an actor or actress..."
)

director_options = (
    unique_people_options(df["imdb_directors"])
    if "imdb_directors" in df.columns
    else []
)
selected_director = st.sidebar.selectbox(
    "Director",
    options=director_options,
    index=None,
    placeholder="Start typing a director..."
)

cinematographer_options = (
    unique_people_options(df["imdb_cinematographers"])
    if "imdb_cinematographers" in df.columns
    else []
)
selected_cinematographer = st.sidebar.selectbox(
    "Cinematographer",
    options=cinematographer_options,
    index=None,
    placeholder="Start typing a cinematographer..."
)


# ──────────────────────────────────────────────────────────────────────────────
# FILTERS
# ──────────────────────────────────────────────────────────────────────────────
filtered_df = df.copy()

if selected_content != "All" and "content_type" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["content_type"] == selected_content]

if selected_source != "All" and "source" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["source"] == selected_source]

if "All" not in selected_languages and selected_languages and "original_language" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["original_language"].isin(selected_languages)]

if year_range and "release_year" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["release_year"].between(year_range[0], year_range[1], inclusive="both")
    ]

if selected_topics:
    filtered_df = filtered_df[
        filtered_df["detected_topics_auto"].apply(
            lambda x: any(topic in x for topic in selected_topics)
        )
    ]

if selected_cast and "imdb_cast" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["imdb_cast"].apply(
            lambda value: row_has_person(value, selected_cast)
        )
    ]

if selected_director and "imdb_directors" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["imdb_directors"].apply(
            lambda value: row_has_person(value, selected_director)
        )
    ]

if selected_cinematographer and "imdb_cinematographers" in filtered_df.columns:
    filtered_df = filtered_df[
        filtered_df["imdb_cinematographers"].apply(
            lambda value: row_has_person(value, selected_cinematographer)
        )
    ]


# ──────────────────────────────────────────────────────────────────────────────
# SIDEBAR QUICK RESULT
# ──────────────────────────────────────────────────────────────────────────────
if sidebar_selected_title:
    sidebar_row = df[df["title"] == sidebar_selected_title].head(1)
    if not sidebar_row.empty:
        row = sidebar_row.iloc[0]

        st.sidebar.markdown("#### Title snapshot")
        st.sidebar.markdown(
            f"**Type:** {row.get('content_type', 'N/A')}  \n"
            f"**Year:** {int(row['release_year']) if pd.notna(row.get('release_year')) else 'N/A'}  \n"
            f"**Language:** {row.get('original_language', 'N/A')}"
        )

        st.sidebar.markdown("#### Overview")
        overview_text = row.get("overview", "")
        if overview_text:
            st.sidebar.caption(overview_text[:350] + ("..." if len(overview_text) > 350 else ""))
        else:
            st.sidebar.caption("No overview available.")

        st.sidebar.markdown("#### Performance")
        st.sidebar.metric("Visibility", metric_str(row.get("visibility_score", np.nan)))
        st.sidebar.metric("Engagement", metric_str(row.get("engagement_score", np.nan)))
        st.sidebar.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))
        if pd.notna(row.get("predicted_business_value", np.nan)):
            st.sidebar.metric("Predicted BV", metric_str(row.get("predicted_business_value", np.nan)))

        st.sidebar.markdown("#### IMDb credits")
        cast_text = row.get("imdb_cast", "")
        director_text = row.get("imdb_directors", "")
        cinematographer_text = row.get("imdb_cinematographers", "")

        st.sidebar.caption(
            f"**Cast:** {cast_text if str(cast_text).strip() else 'N/A'}"
        )
        st.sidebar.caption(
            f"**Director(s):** {director_text if str(director_text).strip() else 'N/A'}"
        )
        st.sidebar.caption(
            f"**Cinematographer(s):** "
            f"{cinematographer_text if str(cinematographer_text).strip() else 'N/A'}"
        )

        active_topics = get_active_topics_from_row(row)
        if not active_topics:
            active_topics = row.get("detected_topics_auto", [])
        st.sidebar.markdown("#### Themes")
        st.sidebar.caption(", ".join(active_topics) if active_topics else "Not detected")


# ──────────────────────────────────────────────────────────────────────────────
# KPI STRIP
# ──────────────────────────────────────────────────────────────────────────────
st.markdown(f"### Filtered catalog: {filtered_df.shape[0]:,} titles")

total_titles = len(filtered_df)
avg_popularity = filtered_df["popularity"].mean() if "popularity" in filtered_df.columns else np.nan
avg_vote = filtered_df["vote_average"].mean() if "vote_average" in filtered_df.columns else np.nan
avg_business = filtered_df["business_value_score"].mean() if "business_value_score" in filtered_df.columns else np.nan
avg_visibility = filtered_df["visibility_score"].mean() if "visibility_score" in filtered_df.columns else np.nan

k1, k2, k3, k4 = st.columns(4)
k1.metric("Total titles", f"{total_titles:,}")
k2.metric("Avg popularity", metric_str(avg_popularity))
k3.metric("Avg rating", metric_str(avg_vote))
k4.metric("Avg business value", metric_str(avg_business))

k5, k6, k7, k8 = st.columns(4)
k5.metric("Avg visibility", metric_str(avg_visibility))
k6.metric("Movies", f"{int((filtered_df['content_type'].astype(str).str.lower() == 'movie').sum()):,}" if "content_type" in filtered_df.columns else "N/A")
k7.metric("Series / TV", f"{int((filtered_df['content_type'].astype(str).str.lower().isin(['tv', 'series'])).sum()):,}" if "content_type" in filtered_df.columns else "N/A")
k8.metric("Sources", f"{filtered_df['source'].nunique():,}" if "source" in filtered_df.columns else "N/A")

st.divider()


# ──────────────────────────────────────────────────────────────────────────────
# TABS
# ──────────────────────────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
    "📊 Executive Overview",
    "🎯 Marketing Performance",
    "🧠 Audience & Themes",
    "🔎 Similar Titles Finder",
    "📝 Synopsis Opportunity Lab",
    "🗂 Dataset Explorer"
])


# ══════════════════════════════════════════════════════════════════════════════
# TAB 1 – EXECUTIVE OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════
with tab1:
    st.subheader("Executive overview")

    c1, c2 = st.columns(2)

    with c1:
        if "content_type" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["content_type"].value_counts()
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_1)
            ax.set_title("Titles by content type", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Count", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "source" in filtered_df.columns and not filtered_df.empty:
            counts = filtered_df["source"].value_counts().head(10)
            fig, ax = make_dark_fig((7, 4))
            ax.bar(counts.index, counts.values, color=ACCENT_2)
            ax.set_title("Top sources", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_ylabel("Count", color=TEXT)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    c3, c4 = st.columns(2)

    with c3:
        if "popularity" in filtered_df.columns and filtered_df["popularity"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["popularity"].dropna(), bins=30, color=ACCENT_3)
            ax.set_title("Popularity distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Popularity", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c4:
        if "business_value_score" in filtered_df.columns and filtered_df["business_value_score"].notna().any():
            fig, ax = make_dark_fig((7, 4))
            ax.hist(filtered_df["business_value_score"].dropna(), bins=30, color=ACCENT_5)
            ax.set_title("Business value distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Business Value Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    if "release_year" in filtered_df.columns and filtered_df["release_year"].notna().any():
        yearly = filtered_df["release_year"].dropna().astype(int).value_counts().sort_index()
        fig, ax = make_dark_fig((12, 4))
        ax.plot(yearly.index, yearly.values)
        ax.set_title("Catalog trend by release year", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        ax.set_xlabel("Release year", color=TEXT)
        ax.set_ylabel("Titles", color=TEXT)
        st.pyplot(fig)
        plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# TAB 2 – MARKETING PERFORMANCE
# ══════════════════════════════════════════════════════════════════════════════
with tab2:
    st.subheader("Marketing performance analysis")

    metric_map = {
        "Visibility": "visibility_score",
        "Engagement": "engagement_score",
        "Business Value": "business_value_score",
        "Popularity": "popularity",
        "Audience Reception": "audience_reception_score"
    }

    available_metric_labels = [k for k, v in metric_map.items() if v in filtered_df.columns]
    selected_metric_label = st.radio(
        "Highlighted KPI",
        options=available_metric_labels,
        horizontal=True,
        index=available_metric_labels.index("Business Value") if "Business Value" in available_metric_labels else 0
    )
    highlight_metric = metric_map[selected_metric_label]

    c1, c2 = st.columns(2)

    with c1:
        top_df = filtered_df[["title", highlight_metric]].dropna().sort_values(
            by=highlight_metric, ascending=False
        ).head(10)

        if not top_df.empty:
            fig, ax = make_dark_fig((8, 5))
            ax.barh(top_df["title"][::-1], top_df[highlight_metric][::-1], color=ACCENT_1)
            ax.set_title(f"Top 10 by {selected_metric_label}", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "marketing_segment" in filtered_df.columns and filtered_df["marketing_segment"].astype(str).str.strip().ne("").any():
            seg = filtered_df["marketing_segment"].replace("", "Unknown").value_counts().head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.bar(seg.index, seg.values, color=ACCENT_2)
            ax.set_title("Marketing segment distribution", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("No `marketing_segment` column available in the current dataset.")

    if all(c in filtered_df.columns for c in ["popularity", "business_value_score"]):
        sample_df = filtered_df[["popularity", "business_value_score"]].dropna().sample(
            min(5000, len(filtered_df[["popularity", "business_value_score"]].dropna())),
            random_state=42
        )
        fig, ax = make_dark_fig((10, 5))
        ax.scatter(sample_df["popularity"], sample_df["business_value_score"], alpha=0.35)
        ax.set_title("Popularity vs Business Value", color=TEXT, fontsize=13, fontweight="bold", pad=10)
        ax.set_xlabel("Popularity", color=TEXT)
        ax.set_ylabel("Business Value Score", color=TEXT)
        st.pyplot(fig)
        plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# TAB 3 – AUDIENCE & THEMES
# ══════════════════════════════════════════════════════════════════════════════
with tab3:
    st.subheader("Audience and thematic signals")

    c1, c2 = st.columns(2)

    with c1:
        topic_counts = {}
        for topic_list in filtered_df["detected_topics_auto"]:
            for topic in topic_list:
                topic_counts[topic] = topic_counts.get(topic, 0) + 1

        if topic_counts:
            topic_series = pd.Series(topic_counts).sort_values(ascending=False).head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.barh(topic_series.index[::-1], topic_series.values[::-1], color=ACCENT_3)
            ax.set_title("Top detected themes", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Count", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

    with c2:
        if "original_language" in filtered_df.columns:
            lang_counts = filtered_df["original_language"].replace("", "Unknown").value_counts().head(10)
            fig, ax = make_dark_fig((8, 5))
            ax.bar(lang_counts.index, lang_counts.values, color=ACCENT_4)
            ax.set_title("Top languages", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.tick_params(axis="x", rotation=25)
            st.pyplot(fig)
            plt.close(fig)

    if "genre_names" in filtered_df.columns:
        exploded = (
            filtered_df.assign(genre_item=filtered_df["genre_list"])
            .explode("genre_item")
        )
        exploded = exploded[exploded["genre_item"].notna() & (exploded["genre_item"] != "")]
        if not exploded.empty:
            top_genres = exploded["genre_item"].value_counts().head(12)
            fig, ax = make_dark_fig((10, 5))
            ax.barh(top_genres.index[::-1], top_genres.values[::-1], color=ACCENT_5)
            ax.set_title("Top genres", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Count", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# TAB 4 – SIMILAR TITLES FINDER
# ══════════════════════════════════════════════════════════════════════════════
with tab4:
    st.subheader("Similar titles finder")

    available_titles = sorted(filtered_df["title"].dropna().astype(str).unique().tolist())

    if not available_titles:
        st.warning("No titles available under the current filters.")
    else:
        selected_title = st.selectbox(
            "Start typing and select a title",
            options=available_titles,
            index=None,
            placeholder="Type a movie or series title..."
        )

        similar_df = pd.DataFrame()

        if selected_title:
            selected_row = filtered_df[filtered_df["title"] == selected_title].head(1)

            if not selected_row.empty:
                row = selected_row.iloc[0]

                rank_pop, total_pop = get_rank(df, selected_title, "popularity")
                rank_vote, total_vote = get_rank(df, selected_title, "vote_average")
                rank_business, total_business = get_rank(df, selected_title, "business_value_score")

                st.markdown(f"#### {selected_title}")

                info1, info2, info3, info4 = st.columns(4)
                info1.metric("Type", row.get("content_type", "N/A"))
                info2.metric("Year", int(row["release_year"]) if pd.notna(row.get("release_year")) else "N/A")
                info3.metric("Rating", metric_str(row.get("vote_average", np.nan)))
                info4.metric("Business Value", metric_str(row.get("business_value_score", np.nan)))

                st.markdown(
                    f"**Genres:** {row.get('genre_names', 'N/A')}  \n"
                    f"**Language:** {row.get('original_language', 'N/A')}  \n"
                    f"**Marketing Segment:** {row.get('marketing_segment', 'N/A')}  \n"
                    f"**Cluster:** {row.get('cluster_label', 'N/A') if str(row.get('cluster_label', '')).strip() else 'N/A'}  \n"
                    f"**Cast:** {row.get('imdb_cast', 'N/A') if str(row.get('imdb_cast', '')).strip() else 'N/A'}  \n"
                    f"**Director(s):** {row.get('imdb_directors', 'N/A') if str(row.get('imdb_directors', '')).strip() else 'N/A'}  \n"
                    f"**Cinematographer(s):** {row.get('imdb_cinematographers', 'N/A') if str(row.get('imdb_cinematographers', '')).strip() else 'N/A'}"
                )

                active_topics = get_active_topics_from_row(row)
                if not active_topics:
                    active_topics = row.get("detected_topics_auto", [])
                st.markdown(f"**Detected themes:** {', '.join(active_topics) if active_topics else 'Not detected'}")

                if row.get("overview"):
                    st.markdown("**Overview**")
                    st.write(row["overview"])

                r1, r2, r3 = st.columns(3)
                r1.metric("Popularity rank", f"#{rank_pop}" if rank_pop else "N/A", delta=f"of {total_pop}" if total_pop else None)
                r2.metric("Rating rank", f"#{rank_vote}" if rank_vote else "N/A", delta=f"of {total_vote}" if total_vote else None)
                r3.metric("Business rank", f"#{rank_business}" if rank_business else "N/A", delta=f"of {total_business}" if total_business else None)

                st.divider()

                similar_df = get_similar_titles(df_similarity, sim_matrix, selected_title, top_n=10)

                st.markdown("#### Most similar titles")

                if similar_df.empty:
                    st.info("No similar titles found.")
                else:
                    fig, ax = make_dark_fig((9, 5))
                    plot_df = similar_df.head(8).copy()
                    ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_1)
                    ax.set_title("Top similarity", color=TEXT, fontsize=13, fontweight="bold", pad=10)
                    ax.set_xlabel("Similarity Score", color=TEXT)
                    st.pyplot(fig)
                    plt.close(fig)

                    display_df = similar_df.copy()
                    if "similarity_score" in display_df.columns:
                        display_df["similarity_score"] = display_df["similarity_score"].round(3)

                    st.dataframe(display_df, use_container_width=True)

        if not similar_df.empty:
            csv_similar = similar_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button(
                label="Download similar titles",
                data=csv_similar,
                file_name="similar_titles_results.csv",
                mime="text/csv"
            )


# ══════════════════════════════════════════════════════════════════════════════
# TAB 5 – SYNOPSIS OPPORTUNITY LAB
# ══════════════════════════════════════════════════════════════════════════════
with tab5:
    st.subheader("Synopsis opportunity lab")
    st.markdown(
        """
        Paste a new synopsis and the app will:
        1. detect its likely themes,
        2. retrieve the most similar titles in the dataset,
        3. estimate a likely business value based on comparable content.
        """
    )

    user_type = st.selectbox(
        "Content type",
        options=["All", "movie", "tv"],
        index=0
    )

    user_synopsis = st.text_area(
        "Paste the synopsis",
        height=180,
        placeholder="Example: A young journalist uncovers a political corruption network while trying to protect her family and save her career..."
    )

    top_n = st.slider("Number of comparables", min_value=3, max_value=10, value=5)

    if st.button("Analyze synopsis"):
        if not user_synopsis or not user_synopsis.strip():
            st.warning("Please paste a synopsis first.")
        else:
            detected_topics = detect_topics_from_synopsis(user_synopsis)

            synopsis_results = get_top_similar_titles_from_synopsis(
                user_synopsis=user_synopsis,
                content_type=user_type,
                df_similarity=df_similarity,
                matrix=sim_matrix,
                top_n=top_n
            )

            estimated_bv = estimate_synopsis_business_value(synopsis_results)
            estimated_band = business_value_band(estimated_bv)

            st.session_state["synopsis_results"] = synopsis_results
            st.session_state["user_synopsis_text"] = user_synopsis
            st.session_state["user_synopsis_topics"] = detected_topics
            st.session_state["user_synopsis_type"] = user_type
            st.session_state["estimated_bv"] = estimated_bv
            st.session_state["estimated_band"] = estimated_band

    if "synopsis_results" in st.session_state:
        synopsis_results = st.session_state["synopsis_results"]
        detected_topics = st.session_state.get("user_synopsis_topics", [])
        user_synopsis_saved = st.session_state.get("user_synopsis_text", "")
        user_type_saved = st.session_state.get("user_synopsis_type", "All")
        estimated_bv = st.session_state.get("estimated_bv", np.nan)
        estimated_band = st.session_state.get("estimated_band", "Unknown")

        a1, a2, a3 = st.columns(3)
        a1.metric("Detected themes", f"{len(detected_topics)}")
        a2.metric("Estimated Business Value", metric_str(estimated_bv))
        a3.metric("Commercial assessment", estimated_band)

        st.markdown("### Detected themes")
        st.write(", ".join(detected_topics) if detected_topics else "No clear themes detected.")

        st.markdown("### Similar titles")
        if synopsis_results.empty:
            st.info("No comparable titles were found for the current filters.")
        else:
            fig, ax = make_dark_fig((9, 5))
            plot_df = synopsis_results.head(8).copy()
            ax.barh(plot_df["title"][::-1], plot_df["similarity_score"][::-1], color=ACCENT_2)
            ax.set_title("Top comparable titles by similarity", color=TEXT, fontsize=13, fontweight="bold", pad=10)
            ax.set_xlabel("Similarity Score", color=TEXT)
            st.pyplot(fig)
            plt.close(fig)

            for i, (_, row) in enumerate(synopsis_results.iterrows(), start=1):
                st.markdown(f"#### #{i} - {row.get('title', 'Unknown title')}")
                st.write(f"**Type:** {row.get('content_type', 'N/A')}")
                st.write(f"**Genres:** {row.get('genre_names', 'N/A')}")
                st.write(f"**Language:** {row.get('original_language', 'N/A')}")
                st.write(f"**Year:** {row.get('release_year', 'N/A')}")
                st.write(f"**Text similarity:** {row.get('text_similarity', 0):.3f}")
                st.write(f"**Topic similarity:** {row.get('topic_similarity', 0):.3f}")
                st.write(f"**Final similarity score:** {row.get('similarity_score', 0):.3f}")

                if "business_value_score" in row and pd.notna(row["business_value_score"]):
                    st.write(f"**Business Value Score:** {row['business_value_score']:.2f}")

                if "predicted_business_value" in row and pd.notna(row["predicted_business_value"]):
                    st.write(f"**Predicted Business Value:** {row['predicted_business_value']:.2f}")

                if "marketing_segment" in row and str(row.get("marketing_segment", "")).strip():
                    st.write(f"**Marketing Segment:** {row.get('marketing_segment')}")

                st.write(f"**Overview:** {row.get('overview', 'N/A')}")
                st.markdown("---")

            st.markdown("### Opportunity readout")
            if pd.notna(estimated_bv):
                if estimated_bv >= 75:
                    st.success(
                        "This synopsis resembles titles with strong commercial potential. "
                        "It may be a good candidate for premium positioning, stronger promotion, or greenlight discussion."
                    )
                elif estimated_bv >= 55:
                    st.info(
                        "This synopsis resembles titles with medium-high potential. "
                        "It could perform well depending on packaging, cast, timing, and campaign strategy."
                    )
                elif estimated_bv >= 40:
                    st.warning(
                        "This synopsis resembles titles with moderate potential. "
                        "It may require sharper positioning or a more differentiated marketing angle."
                    )
                else:
                    st.error(
                        "This synopsis resembles titles with lower estimated commercial value in the current dataset."
                    )
            else:
                st.info("Business value estimation could not be calculated from the available comparable titles.")

            export_df = synopsis_results.copy()
            export_df.insert(0, "input_synopsis", user_synopsis_saved)
            export_df.insert(1, "input_content_type", user_type_saved)
            export_df.insert(2, "detected_topics", ", ".join(detected_topics))
            export_df.insert(3, "estimated_business_value", estimated_bv)
            export_df.insert(4, "estimated_business_value_band", estimated_band)

            csv_synopsis = export_df.to_csv(index=False).encode("utf-8-sig")
            st.download_button(
                label="Download synopsis analysis",
                data=csv_synopsis,
                file_name="synopsis_opportunity_results.csv",
                mime="text/csv"
            )


# ══════════════════════════════════════════════════════════════════════════════
# TAB 6 – DATASET EXPLORER
# ══════════════════════════════════════════════════════════════════════════════
with tab6:
    st.subheader("Dataset explorer")

    explorer_cols = [c for c in [
        "title", "content_type", "genre_names", "release_year",
        "original_language", "source", "popularity", "vote_average",
        "vote_count", "visibility_score", "engagement_score",
        "audience_reception_score", "business_value_score", "predicted_business_value",
        "marketing_segment", "cluster_label",
        "imdb_cast", "imdb_directors", "imdb_cinematographers",
        "detected_topics_text", "overview"
    ] if c in filtered_df.columns]

    st.dataframe(filtered_df[explorer_cols].head(200), use_container_width=True)

    csv = filtered_df.to_csv(index=False).encode("utf-8-sig")
    st.download_button(
        label="Download filtered dataset",
        data=csv,
        file_name="filtered_streaming_dataset.csv",
        mime="text/csv"
    )'''


app_file = Path("streamlit_app.py")
app_file.write_text(streamlit_code, encoding="utf-8")

port = find_free_port()
url = f"http://127.0.0.1:{port}"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(app_file),
    "--server.address=127.0.0.1",
    f"--server.port={port}",
    "--server.headless=true"
])

print(f"Starting Streamlit on {url} ...")

if wait_for_server(url):
    webbrowser.open(url)
    print(f"Streamlit app running at {url}")
else:
    print("Streamlit did not start in time.")
    process.terminate()

Starting Streamlit on http://127.0.0.1:58090 ...



  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:58090

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Streamlit app running at http://127.0.0.1:58090


2026-08-07 02:01:07.955 Pandas DataFrame hash failed. Falling back to pickling the object.
Traceback (most recent call last):
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/streamlit/runtime/caching/hashing.py", line 454, in _to_bytes
    values_hash_bytes = self.to_bytes(hash_pandas_object(df_obj))
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 174, in hash_pandas_object
    h = combine_hash_arrays(hashes, num_items)
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 72, in combine_hash_arrays
    for i, a in enumerate(arrays):
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 173, in <genexpr>
    hashes = (x for x in _hashes)
  File "/Users/blanca/Library/Python/3.9/lib/python/site-packages/pandas/core/util/hashing.py", line 154, in <genexpr>
    hash_array(series._values, encoding, hash_key, categorize)
  File 